# 스마트 도장라인 AI — 단계별 파이프라인 v4

## 이 노트북이 하는 일

컨베이어 도장라인에서 두 가지를 판단합니다.

- **카메라 1** → 후크에 도장 대상이 걸려 있는가? (`LOADED` / `EMPTY`)
- **카메라 2** → 후크의 톱니가 정상적으로 회전하고 있는가? (`정상` / `의심` / `이탈`)

## 전체 파이프라인

```
[섹션 0] 환경 설정
[섹션 1] 이미지 로더 (폴더 / 영상 / 카메라 / 내장)
[섹션 2] YOLO 검출
   ├─ [섹션 2-B] Grounding DINO 자동 라벨링
   ├─ [섹션 2-C] 라벨 검수
   ├─ [섹션 2-D] YOLOv8 파인튜닝
   └─ [섹션 2-E] 성능 검증
[섹션 3] Tracker (ID 연속성)
[섹션 4] 동적 ROI
[섹션 5] Optical Flow
[섹션 6] CNN → GRU 분류
[섹션 7] 후크 상태 큐
[섹션 8] 전체 파이프라인 통합 실행
```

## 실행 순서

| 상황 | 순서 |
|------|------|
| 처음 구조 확인 | 섹션 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 |
| YOLO 파인튜닝 | 섹션 2-B → 2-C → 2-D → 2-E → 섹션 2 재실행 |
| 실데이터 교체 | 섹션 1 MODE 변경 후 전체 재실행 |


---
# 섹션 0. 환경 설정

처음 한 번만 실행하세요. 런타임 재시작 후에도 여기서부터 다시 실행합니다.


In [ ]:
# 섹션 0 — 필수 패키지 설치
# 런타임당 최초 1회 실행. 커널 재시작 후에는 다시 실행하세요.

import subprocess, sys

subprocess.run([sys.executable,"-m","pip","install","-q",
    "ultralytics","lapx","supervision","transformers>=4.38.0","timm"], check=True)
print("패키지 설치 완료")


In [ ]:
# 섹션 0 — 라이브러리 임포트 및 전역 설정
# 디바이스(CPU/GPU), matplotlib 폰트, 핵심 의존성을 초기화합니다.
# ※ 차트 라벨은 깨짐 방지를 위해 영문으로 표기합니다.

import cv2, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.font_manager as fm
import matplotlib
import os, glob, shutil, random, yaml, math, time
from collections import deque
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from pathlib import Path
from PIL import Image as PILImage

# 한글 폰트 (NanumGothic) — 콘솔 출력용. 차트 라벨은 영문만 사용
nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(nanum):
    fe = fm.FontEntry(fname=nanum, name='NanumGothic')
    fm.fontManager.ttflist.insert(0, fe)
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"디바이스: {DEVICE}")
print(f"PyTorch {torch.__version__} | OpenCV {cv2.__version__}")


---
# 섹션 1. 이미지 로더

## 역할

입력 방식(폴더/영상/카메라)이 바뀌어도 이후 섹션 코드는 그대로 써야 합니다.
`MODE` 변수 하나만 바꾸면 됩니다.

| MODE | 설명 |
|------|------|
| `'builtin'` | 내장 테스트 이미지 (현장 영상 없을 때) |
| `'folder'`  | 폴더 안의 jpg/png 파일들 |
| `'video'`   | mp4/avi 영상 파일 |
| `'camera'`  | 웹캠 실시간 |


In [ ]:
# ── 여기만 수정하세요 ──────────────────────────
MODE             = 'builtin'
FOLDER_PATH      = '/content/frames'
FOLDER_EXT       = ('jpg','png','jpeg')
VIDEO_PATH       = '/content/sample.mp4'
VIDEO_SKIP       = 1
CAMERA_ID        = 0
MAX_FRAMES       = 300          # builtin 모드: 생성할 총 프레임 수
FRAME_W          = 640
FRAME_H          = 480
BUILTIN_SAVE_DIR = '/content/builtin_frames'
# ───────────────────────────────────────────────


In [ ]:
# 섹션 1 — 프레임 로더 함수 및 내장 테스트 이미지 생성기
#
# make_builtin_frames()은 현장 영상 없이도 파이프라인 전체를
# 테스트할 수 있도록 도장라인 환경을 합성 렌더링합니다.
#
# 생성 데이터 다양성 (기본 300장):
#   • 다수 후크 — 화면에 최대 3개 동시 표시, 프레임마다 이동
#   • 기어 크기 변형 — 후크마다 크기(18~28px) 랜덤 지정
#   • 회전 패턴 3종 — 정상(일정)/의심(불규칙)/이탈(정지)
#   • 조명 변화 — 프레임마다 배경 밝기 ±30 노이즈 적용
#   • 분진 효과 — 랜덤 백색 점으로 카메라 오염 시뮬레이션

def resize_frame(frame, w, h):
    if w is None or h is None: return frame
    return cv2.resize(frame, (w, h))

def load_frames_from_folder(folder, exts, max_n, w, h):
    paths = sorted([p for p in Path(folder).iterdir()
                    if p.suffix.lower().lstrip('.') in exts])
    if not paths: raise FileNotFoundError(f'이미지 없음: {folder}')
    if max_n: paths = paths[:max_n]
    frames = [resize_frame(cv2.imread(str(p)), w, h)
              for p in paths if cv2.imread(str(p)) is not None]
    print(f'{len(frames)}장 로드: {folder}')
    return frames

def load_frames_from_video(path, skip, max_n, w, h):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): raise FileNotFoundError(f'영상 열기 실패: {path}')
    print(f'영상: 총 {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}프레임')
    frames, idx = [], 0
    while True:
        ret, f = cap.read()
        if not ret: break
        if idx % skip == 0: frames.append(resize_frame(f, w, h))
        idx += 1
        if max_n and len(frames) >= max_n: break
    cap.release()
    print(f'{len(frames)}프레임 추출')
    return frames

def capture_from_camera(cam_id, max_n, w, h):
    cap = cv2.VideoCapture(cam_id)
    if not cap.isOpened(): raise RuntimeError(f'카메라 연결 실패: {cam_id}')
    frames = []
    while len(frames) < (max_n or 999):
        ret, f = cap.read()
        if not ret: break
        frames.append(resize_frame(f, w, h))
    cap.release()
    print(f'캡처 완료: {len(frames)}프레임')
    return frames

def make_builtin_frames(n=300, w=640, h=480):
    """
    도장라인 합성 테스트 프레임 생성기 (기본 300장)

    다양성 요소:
      - 후크 수: 화면에 동시 최대 3개, 프레임마다 이동 (간격 180px)
      - 기어 크기: 후크마다 18~28px 랜덤
      - 회전 패턴:
          NORMAL   — 매 프레임 일정 각도(7°) 회전  (정상)
          SUSPECT  — 각도가 불규칙하게 변함          (의심)
          DERAILED — 회전 정지 + 미세 진동만         (이탈)
      - 조명: 배경 기본 밝기 ±25 가우시안 노이즈
      - 분진: 프레임마다 랜덤 위치에 흰 점 0~80개
      - 다중 후크 위치: 각 후크는 독립된 x 시작점에서 출발
    """
    frames = []

    rack_y  = int(h * 0.60)
    chain_y = int(h * 0.14)

    # 후크 설정: (시작 x, 기어 반지름, 회전 패턴, 초기각도)
    # 패턴: 0=NORMAL, 1=SUSPECT, 2=DERAILED
    hook_configs = []
    n_hooks = 3
    for hi in range(n_hooks):
        start_x  = int(w * 0.12) + hi * 185
        gear_r   = random.randint(18, 28)
        pattern  = hi % 3          # 0,1,2 골고루
        init_ang = random.uniform(0, 360)
        hook_configs.append({'x': start_x, 'r': gear_r,
                             'pattern': pattern, 'angle': init_ang})

    speed_px = 5.2   # 컨베이어 이동 속도 (px/프레임)

    for i in range(n):
        # ── 배경 조명 변화 ──
        base_bright = 55 + int(random.gauss(0, 12))
        base_bright = max(30, min(80, base_bright))
        frame = np.full((h, w, 3), base_bright, dtype=np.uint8)

        # ── 바닥/레일/체인 ──
        cv2.rectangle(frame,(0,rack_y+28),(w,h),(40,40,38),-1)
        cv2.rectangle(frame,(0,chain_y-4),(w,chain_y+10),(90,90,88),-1)
        cv2.rectangle(frame,(0,chain_y-4),(w,chain_y),(110,110,108),-1)
        for rx in range(20, w, 32):
            cv2.ellipse(frame,(rx,chain_y+3),(12,5),0,0,360,(70,70,68),-1)
            cv2.ellipse(frame,(rx,chain_y+3),(12,5),0,0,360,(50,50,48),1)
        cv2.rectangle(frame,(0,rack_y),(w,rack_y+28),(75,75,72),-1)
        cv2.rectangle(frame,(0,rack_y-2),(w,rack_y+4),(95,95,92),-1)
        cv2.rectangle(frame,(0,rack_y+24),(w,rack_y+28),(55,55,52),-1)
        for tx in range(8, w, 14):
            cv2.rectangle(frame,(tx,rack_y-14),(tx+8,rack_y+2),(62,62,60),-1)

        # ── 각 후크 렌더링 ──
        for hc in hook_configs:
            cx_now = int(hc['x'] + i * speed_px) % (w + 300) - 150
            if cx_now < -60 or cx_now > w + 60:
                continue   # 화면 밖이면 스킵

            arm_top = chain_y + 10
            arm_bot = rack_y - 28
            cv2.rectangle(frame,(cx_now-9,arm_top),(cx_now+9,arm_bot),(80,78,75),-1)
            cv2.rectangle(frame,(cx_now-14,arm_bot-4),(cx_now+14,arm_bot+6),(85,83,80),-1)

            gear_cx = cx_now
            gear_cy = int(rack_y - 20)
            gear_r  = hc['r']

            cv2.ellipse(frame,(gear_cx,gear_cy),(gear_r,int(gear_r*0.55)),0,0,360,(95,92,88),-1)
            cv2.ellipse(frame,(gear_cx,gear_cy),(gear_r,int(gear_r*0.55)),0,0,360,(60,58,55),1)

            # 회전 패턴별 각도 증가
            pattern = hc['pattern']
            if pattern == 0:          # NORMAL: 일정 회전
                hc['angle'] += 7.0
            elif pattern == 1:        # SUSPECT: 불규칙 회전
                hc['angle'] += random.uniform(0, 14.0)
            else:                     # DERAILED: 정지 + 미세 진동
                hc['angle'] += random.uniform(-0.5, 0.5)

            angle = hc['angle']
            tooth_count = 8
            for k in range(tooth_count):
                theta = math.radians(angle + k * (360 / tooth_count))
                pts = np.array([
                    [int(gear_cx+(gear_r-2)*math.cos(theta-0.22)),
                     int(gear_cy+(gear_r-2)*0.55*math.sin(theta-0.22))],
                    [int(gear_cx+(gear_r+9)*math.cos(theta)),
                     int(gear_cy+(gear_r+9)*0.55*math.sin(theta))],
                    [int(gear_cx+(gear_r-2)*math.cos(theta+0.22)),
                     int(gear_cy+(gear_r-2)*0.55*math.sin(theta+0.22))],
                ], dtype=np.int32)
                cv2.fillPoly(frame,[pts],(108,105,100))
                cv2.polylines(frame,[pts],True,(55,53,50),1)

            cv2.ellipse(frame,(gear_cx,gear_cy),(8,5),0,0,360,(55,53,50),-1)
            cv2.ellipse(frame,(gear_cx,gear_cy),(3,2),0,0,360,(40,38,35),-1)

            # 패턴 레이블 표시
            label = ['N','S','D'][pattern]
            cv2.putText(frame,f'H{hook_configs.index(hc)} {label}',
                        (max(0,cx_now-18), gear_cy-gear_r-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.38,(180,178,172),1)

        # ── 분진 효과 (랜덤 흰 점) ──
        n_dust = random.randint(0, 80)
        if n_dust > 0:
            dust_x = np.random.randint(0, w, n_dust)
            dust_y = np.random.randint(0, h, n_dust)
            for dx, dy in zip(dust_x, dust_y):
                cv2.circle(frame,(int(dx),int(dy)),1,(200,200,200),-1)

        # ── 프레임 번호 ──
        cv2.putText(frame,f'Frame {i:03d}',(10,24),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(160,160,155),1)
        frames.append(frame)

    return frames

if MODE == 'folder':
    FRAMES = load_frames_from_folder(FOLDER_PATH,FOLDER_EXT,MAX_FRAMES,FRAME_W,FRAME_H)
elif MODE == 'video':
    FRAMES = load_frames_from_video(VIDEO_PATH,VIDEO_SKIP,MAX_FRAMES,FRAME_W,FRAME_H)
elif MODE == 'camera':
    FRAMES = capture_from_camera(CAMERA_ID,MAX_FRAMES,FRAME_W,FRAME_H)
else:
    FRAMES = make_builtin_frames(n=MAX_FRAMES, w=FRAME_W, h=FRAME_H)
    os.makedirs(BUILTIN_SAVE_DIR,exist_ok=True)
    for i,f in enumerate(FRAMES):
        cv2.imwrite(os.path.join(BUILTIN_SAVE_DIR,f'frame_{i:03d}.jpg'),f)
    print(f'내장 이미지 {len(FRAMES)}장 저장: {BUILTIN_SAVE_DIR}')
    print('Colab 왼쪽 파일탐색기 → content → builtin_frames 에서 확인')

print(f'\n로드 완료: {len(FRAMES)}프레임  크기={FRAMES[0].shape}')


In [ ]:
# 섹션 1 — 로드된 프레임 확인 (적응형 시각화)
#
# 표시 장수는 총 프레임 수에 비례해 자동 결정됩니다.
#   50장 미만  → 5장  /  50~149장 → 10장  /  150장 이상 → 15장
#
# 2개 패널로 구성됩니다:
#   [패널 A] 전체 균등 샘플  — 이미지 품질·후크 가시성 확인
#   [패널 B] 연속 구간 8장   — 프레임 간 이동 흐름(연속성) 확인
#
# ※ builtin 프레임은 파이프라인 구조 동작 확인용입니다.
#    실제 회전 상태(정상/의심/이탈) 판정은 현장 영상 교체 후 유효합니다.

import math as _math

n = len(FRAMES)

# ── 표시 장수 자동 결정 ──
if n < 50:
    n_sample = 5
elif n < 150:
    n_sample = 10
else:
    n_sample = 15

n_cols = 5
n_rows = _math.ceil(n_sample / n_cols)

# ══════════════════════════════════
# 패널 A: 전체 균등 샘플
# ══════════════════════════════════
sample_idxs = [int(i * (n - 1) / (n_sample - 1)) for i in range(n_sample)]

fig_a, axes_a = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 3.2, n_rows * 2.8))
axes_a = np.array(axes_a).flatten()

for ax, idx in zip(axes_a, sample_idxs):
    ax.imshow(cv2.cvtColor(FRAMES[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx}', fontsize=8)
    ax.axis('off')

for ax in axes_a[len(sample_idxs):]:
    ax.axis('off')

fig_a.suptitle(
    f'[Section 1 - Panel A] Uniform Sample  |  '
    f'Total={n} frames  n_sample={n_sample}  Size={FRAMES[0].shape}',
    fontsize=11, fontweight='bold')
fig_a.tight_layout()
plt.show()

# ══════════════════════════════════
# 패널 B: 연속 구간 8장 (중간 지점 기준)
# — 카메라 1이 후크를 감지하고 카메라 2가
#   판정하는 흐름을 시뮬레이션하려면
#   후크가 프레임 간에 연속적으로 이동해야 합니다.
#   이 패널에서 그 연속성을 눈으로 확인하세요.
# ══════════════════════════════════
SEQ_LEN   = 8
seq_start = max(0, n // 2 - SEQ_LEN // 2)
seq_idxs  = list(range(seq_start, min(n, seq_start + SEQ_LEN)))

fig_b, axes_b = plt.subplots(1, len(seq_idxs),
                              figsize=(len(seq_idxs) * 3.2, 3.4))
if len(seq_idxs) == 1:
    axes_b = [axes_b]

for ax, idx in zip(axes_b, seq_idxs):
    ax.imshow(cv2.cvtColor(FRAMES[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx}', fontsize=8)
    ax.axis('off')

fig_b.suptitle(
    f'[Section 1 - Panel B] Consecutive Sequence  |  Frames {seq_idxs[0]}–{seq_idxs[-1]}\n'
    f'Cam1: hook load detection  →  conveyor travel  →  Cam2: gear rotation judgement',
    fontsize=10, fontweight='bold')
fig_b.tight_layout()
plt.show()

# ── 콘솔 요약 ──
print(f'로드 완료: {n}프레임  크기={FRAMES[0].shape}')
print(f'샘플 표시: {n_sample}장  (n < 50 → 5장 / 50~149 → 10장 / 150+ → 15장)')
print()
print('확인 체크리스트:')
print('  [패널 A] □ 이미지 정상  □ 후크/톱니 화면 안에 있음')
print('  [패널 B] □ 후크가 프레임 간 연속적으로 이동함  □ 갑작스러운 끊김 없음')
print()
print('※ builtin 모드: 파이프라인 구조 동작 확인용 합성 이미지입니다.')
print('   실제 판정 신뢰도는 현장 영상(MODE=folder/video/camera) 교체 후 평가하세요.')


---
# 섹션 2. YOLO 검출

## 역할

Optical Flow 계산 전에 **톱니가 어디 있는지** 먼저 알아야 합니다.
컨베이어 이동으로 프레임마다 위치가 달라지므로 YOLO가 매 프레임 bbox를 제공합니다.

| YOLO_MODE | 상황 |
|-----------|------|
| `'pretrained'` | 파인튜닝 전 구조 확인용 (gear/hook 미지원) |
| `'custom'` | 섹션 2-D 완료 후 best.pt 경로 입력 |
| `'manual_roi'` | YOLO 없이 좌표 직접 지정 |

> gear/hook 파인튜닝이 필요하면 **섹션 2-B부터** 먼저 진행하세요.


In [ ]:
# ── 여기만 수정하세요 ──────────────────────────

# YOLO_MODE: 검출 방식 선택
#   'pretrained'  → COCO 사전학습 모델 사용 (gear/hook 클래스 없음, 구조 확인용)
#   'custom'      → 섹션 2-D에서 파인튜닝한 best.pt 사용 (실제 운용)
#   'manual_roi'  → YOLO 없이 고정 좌표를 직접 지정 (임시 테스트)
YOLO_MODE    = 'custom'

# YOLO_WEIGHTS: 모델 파일 경로
#   pretrained 모드 → 'yolov8s.pt' (자동 다운로드)
#   custom 모드    → 섹션 2-D 완료 후 생성된 best.pt 경로로 교체
YOLO_WEIGHTS = 'runs/detect/runs/finetune/coating_line/weights/best.pt'

# YOLO_CONF: 검출 신뢰도 임계값 (0.0~1.0)
#   이 값 이상의 확신도를 가진 bbox만 유효한 검출로 인정합니다.
#   높을수록 오탐↓ 미검출↑ / 낮을수록 오탐↑ 미검출↓
#   권장: 파인튜닝 완료 후 0.60~0.70 / 구조 확인 중에는 0.30~0.50
YOLO_CONF    = 0.65

# YOLO_CLASSES: 검출할 클래스 ID 목록
#   pretrained 모드 → None (COCO 전체 클래스)
#   custom 모드    → [0, 1] (0=gear, 1=hook)
YOLO_CLASSES = [0, 1]

# MANUAL_ROI: manual_roi 모드에서 사용할 고정 관심 영역 (픽셀 좌표)
#   형식: (x1, y1, x2, y2) — 좌상단·우하단 꼭짓점
#   톱니가 항상 같은 위치에 있을 때만 유효합니다.
MANUAL_ROI   = (200, 160, 440, 360)

# ───────────────────────────────────────────────


In [ ]:
from ultralytics import YOLO as _YOLO

yolo_model = None

if YOLO_MODE in ('pretrained','custom'):
    weights    = 'yolov8s.pt' if YOLO_MODE == 'pretrained' else YOLO_WEIGHTS
    yolo_model = _YOLO(weights)
    print(f'YOLO 로드: {weights}')
    if YOLO_MODE == 'pretrained':
        print('※ COCO 사전학습 — gear/hook 클래스 없음')
        print('  검출 안 되면 정상입니다. 섹션 2-B에서 파인튜닝하거나')
        print('  YOLO_MODE = manual_roi 로 임시 진행하세요.')
else:
    print(f'수동 ROI 모드: {MANUAL_ROI}')

def detect_objects(frame: np.ndarray) -> List[Dict]:
    if YOLO_MODE == 'manual_roi':
        return [{'bbox':MANUAL_ROI,'conf':1.0,'cls':0}]
    results = yolo_model(frame, conf=YOLO_CONF,
                          classes=YOLO_CLASSES, verbose=False)
    return [{'bbox':tuple(int(v) for v in box.xyxy[0].tolist()),
             'conf':float(box.conf),'cls':int(box.cls)}
            for box in results[0].boxes]


In [ ]:
# 섹션 2 — YOLO 검출 결과 시각화
# 균등 간격 6프레임을 샘플링해 검출 박스를 그립니다.
# 주황=gear(cls0), 파랑=hook(cls1). 전체 검출률도 계산합니다.
# ※ 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

check_idxs = [int(i*(len(FRAMES)-1)/5) for i in range(6)]
fig, axes = plt.subplots(2,3,figsize=(15,8)); axes=axes.flatten()

for ax, idx in zip(axes, check_idxs):
    frame = FRAMES[idx]
    dets  = detect_objects(frame)
    vis   = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB)
    for d in dets:
        x1,y1,x2,y2 = d['bbox']
        color = '#EF9F27' if d['cls']==0 else '#185FA5'
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec=color,fc='none'))
        ax.text(x1,y1-4,f"cls{d['cls']} {d['conf']:.2f}",color=color,fontsize=8,fontweight='bold')
    ax.imshow(vis)
    ax.set_title(f'Frame {idx}  |  {len(dets)} detection(s)',fontsize=10); ax.axis('off')
    if not dets:
        ax.text(0.5,0.5,'No detection',transform=ax.transAxes,
                ha='center',va='center',fontsize=13,color='red')

plt.suptitle('[Section 2] YOLO Detection  |  Orange=gear(cls0)  Blue=hook(cls1)',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

det_rate = sum(1 for f in FRAMES if detect_objects(f))
print(f'전체 검출률: {det_rate}/{len(FRAMES)} ({det_rate/len(FRAMES)*100:.1f}%)')
print()
print('검출이 안 된다면:')
print('  → YOLO_CONF를 낮추세요 (예: 0.1)')
print('  → 섹션 2-B에서 파인튜닝 진행')
print('  → YOLO_MODE = manual_roi 로 임시 진행')


---
# 섹션 2-B. YOLO 파인튜닝 — Grounding DINO 자동 라벨링

## 역할

COCO 모델은 `gear`/`hook`을 모릅니다.
현장 이미지로 파인튜닝해야 분진·조명 등 현장 환경에서 정확하게 작동합니다.

**Grounding DINO**: 텍스트 프롬프트 `"gear . hook"` 만으로 bbox 자동 생성 → YOLO 라벨로 변환

## 클래스 정의

| ID | 클래스 | 설명 |
|----|--------|------|
| 0  | `gear` | 톱니바퀴 |
| 1  | `hook` | 후크 암 |

> 이미 `best.pt`가 있으면 이 섹션 건너뛰고 섹션 2에서 `YOLO_MODE='custom'` 설정


In [ ]:
# 섹션 2-B — Grounding DINO 설정
#
# Grounding DINO(그라운딩 다이노)는 텍스트 설명만으로 물체를 찾는
# 제로샷(zero-shot) 객체 검출 모델입니다.
# 별도 학습 없이 자연어 프롬프트로 bbox를 자동 생성합니다.
#
# ── 프롬프트 작성 규칙 ──────────────────────────────────────────
#  • 영문 명사/명사구 위주로 작성 (긴 문장보다 핵심 단어가 더 정확)
#  • 여러 대상은 " . "(점 + 공백)으로 구분
#  • 너무 추상적이거나 동사 위주 문장은 검출률 저하
#  • 아래 후보 중 섹션 2-B 진단 셀(바로 아래)에서 성능을 비교하세요.
#
# ── 프롬프트 후보 (성능은 이미지마다 다름, 진단 셀에서 비교 권장) ──
#
#  [짧은 버전] — 일반적인 경우
#    "gear"
#    "gear . hook"
#
#  [형태 묘사 버전] — 톱니 모양을 더 구체적으로
#    "sprocket gear . gear tooth . gear wheel"
#
#  [맥락 추가 버전] — 도장라인/산업 환경 명시
#    "metal sprocket gear . rotating gear wheel . industrial conveyor gear"
#
#  [hook 포함 버전] — gear + hook 동시 검출 시
#    "metal sprocket gear . gear tooth . conveyor hook . hook arm"
#
# ── 여기만 수정하세요 ──────────────────────────

# 사용할 프롬프트 — 진단 셀 실행 후 가장 탐지 수가 많은 것으로 교체
GDINO_PROMPT = "metal sprocket gear . gear tooth . gear wheel"

# CLASS_MAP: 프롬프트의 어떤 단어를 어떤 클래스 ID로 매핑할지 지정
#   키(key)가 프롬프트에 포함된 단어와 일치해야 합니다.
CLASS_MAP = {"gear": 0}          # hook도 검출하려면: {"gear": 0, "hook": 1}

# GDINO_BOX_THRESHOLD: bbox 신뢰도 임계값 (0.0~1.0)
#   높을수록 확실한 박스만 남김 (오탐↓ 미검출↑)
#   낮을수록 더 많이 탐지       (오탐↑ 미검출↓)
#   권장: 0.30~0.40  /  탐지가 너무 없으면 0.10~0.20으로 낮추세요
GDINO_BOX_THRESHOLD  = 0.35

# GDINO_TEXT_THRESHOLD: 텍스트-이미지 유사도 임계값
#   GDINO_BOX_THRESHOLD와 함께 작동하는 보조 필터입니다.
#   일반적으로 BOX보다 낮게 설정합니다.
GDINO_TEXT_THRESHOLD = 0.15

# UPLOAD_MODE: True → Colab 파일 업로드 / False → 폴더 경로 사용
UPLOAD_MODE        = False

# GDINO_IMAGE_FOLDER: UPLOAD_MODE=False 일 때 이미지를 읽어올 폴더 경로
GDINO_IMAGE_FOLDER = '/content/builtin_frames'

# ───────────────────────────────────────────────

os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)


In [ ]:
# 이미지 업로드 또는 폴더 로드
if UPLOAD_MODE:
    from google.colab import files
    print("라벨링할 현장 이미지를 업로드하세요 (여러 장 선택 가능)")
    uploaded = files.upload()
    image_paths = []
    for fname, data in uploaded.items():
        sp = f"dataset/images/train/{fname}"
        with open(sp,"wb") as f: f.write(data)
        image_paths.append(sp)
    print(f"\n{len(image_paths)}장 업로드 완료")
else:
    image_paths = (glob.glob(f"{GDINO_IMAGE_FOLDER}/*.jpg") +
                   glob.glob(f"{GDINO_IMAGE_FOLDER}/*.png"))
    print(f"{len(image_paths)}장 발견")


In [ ]:
# Grounding DINO 로드 (transformers 버전 — 안정적)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torch as _torch

GDINO_REPO = "IDEA-Research/grounding-dino-base"
device_gdino = "cuda" if _torch.cuda.is_available() else "cpu"

print(f"Grounding DINO 로드 중... (첫 실행 시 모델 다운로드 약 1GB)")
gdino_processor = AutoProcessor.from_pretrained(GDINO_REPO)
gdino_model_hf  = AutoModelForZeroShotObjectDetection.from_pretrained(
    GDINO_REPO).to(device_gdino)

print(f"✅ Grounding DINO 로드 완료  디바이스: {device_gdino}")


In [ ]:
# bbox 시각화 확인 — 실제 추론 전에 한 장으로 먼저 확인
# NMS로 중복 박스 제거 + 이미지 위 텍스트는 영문 표기

from PIL import Image as _PIL
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch as _torch
import numpy as np

device_gdino = "cuda" if _torch.cuda.is_available() else "cpu"

# 확인할 이미지 번호 (변경 가능)
CHECK_IDX = 0

def nms_boxes(boxes, scores, iou_threshold=0.5):
    """중복 박스 제거 — 같은 위치를 여러 번 감지한 경우 점수 높은 것만 유지"""
    if len(boxes) == 0:
        return []
    boxes_arr  = np.array(boxes,  dtype=np.float32)
    scores_arr = np.array(scores, dtype=np.float32)
    order = scores_arr.argsort()[::-1]  # 점수 높은 순서로 정렬
    keep  = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        xx1 = np.maximum(boxes_arr[i,0], boxes_arr[rest,0])
        yy1 = np.maximum(boxes_arr[i,1], boxes_arr[rest,1])
        xx2 = np.minimum(boxes_arr[i,2], boxes_arr[rest,2])
        yy2 = np.minimum(boxes_arr[i,3], boxes_arr[rest,3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        area_i    = (boxes_arr[i,2]-boxes_arr[i,0]) * (boxes_arr[i,3]-boxes_arr[i,1])
        area_rest = ((boxes_arr[rest,2]-boxes_arr[rest,0]) *
                     (boxes_arr[rest,3]-boxes_arr[rest,1]))
        iou   = inter / (area_i + area_rest - inter + 1e-6)
        order = rest[iou < iou_threshold]
    return keep

# ── 추론 실행 ──
pil_img = _PIL.open(image_paths[CHECK_IDX]).convert("RGB")
W, H    = pil_img.size

text   = GDINO_PROMPT.replace(".", ". ").strip() + "."
inputs = gdino_processor(images=pil_img, text=text,
                          return_tensors="pt").to(device_gdino)
with _torch.no_grad():
    outputs = gdino_model_hf(**inputs)

res = gdino_processor.post_process_grounded_object_detection(
    outputs, inputs.input_ids,
    threshold      = GDINO_BOX_THRESHOLD,
    text_threshold = GDINO_TEXT_THRESHOLD,
    target_sizes   = [(H, W)]
)[0]

# ── 클래스 매핑 ──
CLASS_COLOR = {"gear": "#EF9F27", "hook": "#185FA5"}
DEFAULT_CLR = "#A32D2D"

raw_boxes, raw_scores, raw_cls, raw_colors = [], [], [], []
for score, label, box in zip(res["scores"], res["text_labels"], res["boxes"]):
    label_str = label.lower()
    if "gear" in label_str:
        cls_name = "gear";  color = CLASS_COLOR["gear"]
    elif "hook" in label_str:
        cls_name = "hook";  color = CLASS_COLOR["hook"]
    else:
        cls_name = "other"; color = DEFAULT_CLR
    raw_boxes.append(box.tolist())
    raw_scores.append(float(score))
    raw_cls.append(cls_name)
    raw_colors.append(color)

# ── 클래스별 NMS 적용 ──
# final_indices = []
# for cls_name in ["gear", "hook", "other"]:
#     idx = [i for i, c in enumerate(raw_cls) if c == cls_name]
#     if not idx:
#         continue
#     kept = nms_boxes([raw_boxes[i]  for i in idx],
#                      [raw_scores[i] for i in idx],
#                      iou_threshold=0.5)
#     final_indices.extend([idx[k] for k in kept])

# ── 시각화 ──
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(pil_img)

n_gear = n_hook = n_other = 0
for i in sorted(final_indices):
    x1, y1, x2, y2 = raw_boxes[i]
    score    = raw_scores[i]
    cls_name = raw_cls[i]
    color    = raw_colors[i]

    # bbox 사각형
    rect = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2, edgecolor=color, facecolor="none"
    )
    ax.add_patch(rect)

    # 이미지 위 라벨 — 영문만 사용 (한글 폰트 미지원으로 깨짐 방지)
    ax.text(
        x1, max(y1 - 5, 10),
        f"{cls_name} {score:.2f}",
        color="white", fontsize=9, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85)
    )
    if cls_name == "gear":   n_gear  += 1
    elif cls_name == "hook": n_hook  += 1
    else:                    n_other += 1

total = n_gear + n_hook + n_other

# 그래프 제목 — 영문만 사용
ax.set_title(
    f"{image_paths[CHECK_IDX]}\n"
    f"prompt='{GDINO_PROMPT}'  box_thr={GDINO_BOX_THRESHOLD}  text_thr={GDINO_TEXT_THRESHOLD}  NMS=0.5\n"
    f"detected: gear={n_gear}  hook={n_hook}  other={n_other}  total={total}",
    fontsize=10
)
ax.axis("off")
plt.tight_layout()
plt.show()

# 콘솔 출력은 한글 사용 가능
print(f"NMS 적용 후: gear={n_gear}개  hook={n_hook}개  기타={n_other}개  합계={total}개")
print()

if total > 5:
    print(f"⚠ 탐지 {total}개 — 오탐 가능성 있음")
    print(f"  GDINO_BOX_THRESHOLD를 높이세요 (현재: {GDINO_BOX_THRESHOLD})")
    print(f"  예: 0.30 → 0.35 → 0.40 순으로 올려가며 재실행")
elif total == 0:
    print("⚠ 탐지 0개 — GDINO_BOX_THRESHOLD를 낮추세요")
else:
    print(f"✅ 탐지 {total}개 — bbox 위치를 눈으로 확인하세요")

print()
print("확인 사항:")
print("  □ bbox가 실제 gear/hook 위치를 감싸고 있는가?")
print("  □ 배경, 레일, 체인이 오탐되지 않았는가?")
print()
print("다음 단계:")
print("  bbox 위치 맞음 → 추론 셀(run_gdino) 실행")
print("  bbox 위치 틀림 → GDINO_BOX_THRESHOLD 조정 후 이 셀 재실행")
print(f"  다른 이미지 확인 → CHECK_IDX 변경 (0 ~ {len(image_paths)-1})")

In [ ]:
# 섹션 2-B — 탐지 전 진단: 임계값별·프롬프트별 탐지 수 비교
#
# 이 셀을 가장 먼저 실행해 최적 프롬프트와 임계값을 찾으세요.
# 결과를 바탕으로 위 설정 셀의 GDINO_PROMPT와 GDINO_BOX_THRESHOLD를 수정합니다.
#
# Grounding DINO 프롬프트 선택 기준:
#   1) 탐지 수가 너무 많으면(5개↑) → 임계값을 높이거나 프롬프트를 구체화
#   2) 탐지 수가 0이면            → 임계값을 낮추거나 더 일반적인 프롬프트 시도
#   3) 탐지 수 1~3개가 이상적     → 박스 위치를 아래 시각화로 확인

if not image_paths:
    print("image_paths 가 비어 있습니다. 이미지 업로드/로드 셀을 먼저 실행하세요.")
else:
    test_img = PILImage.open(image_paths[0]).convert("RGB")
    W_t, H_t = test_img.size
    print(f"진단 이미지: {image_paths[0]}  ({W_t}x{H_t})")
    print()

    # ── 1) 임계값별 탐지 수 ──────────────────────────────────────
    print("── 임계값별 탐지 수 (현재 프롬프트로 비교) ───────────────────")
    print(f"   현재 프롬프트: '{GDINO_PROMPT}'")
    print()
    for box_th, text_th, label in [
        (0.40, 0.20, "엄격  — 오탐 최소, 미검출 가능"),
        (0.30, 0.15, "권장  — 균형"),
        (0.15, 0.10, "완화  — 탐지↑ 오탐↑"),
        (0.05, 0.05, "최저  — 노이즈 많음, 참고용"),
    ]:
        text   = GDINO_PROMPT.replace(".", ". ").strip() + "."
        inputs = gdino_processor(images=test_img, text=text,
                                  return_tensors="pt").to(device_gdino)
        with _torch.no_grad():
            outputs = gdino_model_hf(**inputs)
        res = gdino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold=box_th, text_threshold=text_th,
            target_sizes=[(H_t, W_t)]
        )[0]
        n      = len(res["scores"])
        scores = [f"{s:.2f}" for s in res["scores"].tolist()]
        flag   = "  ← 권장 범위" if 1 <= n <= 3 else ("  ← 오탐 의심" if n > 3 else "  ← 미검출")
        print(f"  box={box_th}  text={text_th}  [{label}] → {n}개  점수: {scores}{flag}")

    print()

    # ── 2) 프롬프트별 탐지 수 ────────────────────────────────────
    print("── 프롬프트별 탐지 수 비교 (임계값 box=0.15 고정) ────────────")
    print("   짧을수록 범용 / 길수록 구체적 — 현장 이미지에 따라 결과 다름")
    print()

    # 프롬프트 후보: (프롬프트, 설명)
    test_prompts = [
        # 단어 수준
        ("gear",                                                "단어 — 가장 범용"),
        ("gear . hook",                                         "단어 — gear+hook 동시"),
        # 형태 묘사
        ("sprocket gear . gear tooth . gear wheel",             "형태 묘사 — 톱니 특징"),
        ("metal sprocket gear . gear tooth . gear wheel",       "형태+재질 묘사"),
        # 도장라인 맥락
        ("rotating gear . conveyor hook",                       "맥락 — 회전/컨베이어"),
        ("metal sprocket gear . rotating gear wheel . industrial conveyor gear",
                                                                "맥락+형태 — 산업용 기어"),
        # hook 포함
        ("metal sprocket gear . gear tooth . conveyor hook . hook arm",
                                                                "gear+hook 동시, 상세"),
        # 단순 조합
        ("industrial gear . industrial hook",                   "산업용 단순 조합"),
        ("gear wheel . hook arm",                               "바퀴/암 표현"),
    ]

    best_prompt, best_n = GDINO_PROMPT, 0
    results_table = []

    for prompt, desc in test_prompts:
        text   = prompt.replace(".", ". ").strip() + "."
        inputs = gdino_processor(images=test_img, text=text,
                                  return_tensors="pt").to(device_gdino)
        with _torch.no_grad():
            outputs = gdino_model_hf(**inputs)
        res = gdino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold=0.15, text_threshold=0.10,
            target_sizes=[(H_t, W_t)]
        )[0]
        n      = len(res["scores"])
        status = "✅" if 1 <= n <= 3 else ("⚠️" if n > 3 else "❌")
        scores = [f"{s:.2f}" for s in res["scores"].tolist()]
        results_table.append((status, n, prompt, desc, scores))
        if n > best_n:
            best_n, best_prompt = n, prompt

    # 탐지 수 내림차순 출력
    for status, n, prompt, desc, scores in sorted(results_table,
                                                   key=lambda x: -x[1]):
        print(f"  {status} {n}개  {desc}")
        print(f"       프롬프트: \"{prompt}\"")
        print(f"       점수: {scores}")
        print()

    # ── 3) 추천 결과 ──────────────────────────────────────────────
    print("=" * 58)
    if best_n > 0:
        # 1~3개인 것 중 최적 찾기
        good = [(n, p, d) for _, n, p, d, _ in results_table if 1 <= n <= 3]
        if good:
            best_n, best_prompt, best_desc = sorted(good, key=lambda x: -x[0])[0]
            print(f"추천 프롬프트: \"{best_prompt}\"")
            print(f"  → 탐지 수: {best_n}개  ({best_desc})")
        else:
            print(f"탐지 수가 가장 많은 프롬프트: \"{best_prompt}\" ({best_n}개)")
            print(f"  ⚠ 탐지가 너무 많습니다. BOX 임계값을 높이거나")
            print(f"    더 구체적인 프롬프트로 교체 후 재실행하세요.")
        print()
        print("설정 셀(위)에서 아래 값을 수정 후 재실행하세요:")
        print(f'  GDINO_PROMPT        = "{best_prompt}"')
        print(f"  GDINO_BOX_THRESHOLD = 0.30   # 권장 시작값, 조정 가능")
    else:
        print("❌ 모든 프롬프트에서 탐지 실패")
        print()
        print("원인 및 대안:")
        print("  1) 임계값을 더 낮추세요: GDINO_BOX_THRESHOLD = 0.05")
        print("  2) 현장 특수 부품이라 DINO가 시각적으로 인식 어려울 수 있음")
        print("  3) 수동 라벨링으로 전환:")
        print("       https://www.makesense.ai  (웹, 무료)")
        print("       labelImg                  (로컬 설치)")
        print("     포맷: YOLO  저장 위치: dataset/labels/train/")
    print("=" * 58)

    # ── 4) 진단 이미지 시각화 ─────────────────────────────────────
    plt.figure(figsize=(7, 5))
    plt.imshow(test_img)
    plt.title(f"Diagnosis image\nprompt: '{GDINO_PROMPT}'  box_thr={GDINO_BOX_THRESHOLD}")
    plt.axis('off'); plt.tight_layout(); plt.show()


In [ ]:
# 현재 설정값 확인
print(f"GDINO_PROMPT         = '{GDINO_PROMPT}'")
print(f"GDINO_BOX_THRESHOLD  = {GDINO_BOX_THRESHOLD}")
print(f"GDINO_TEXT_THRESHOLD = {GDINO_TEXT_THRESHOLD}")

In [ ]:
# 섹션 2-B — Grounding DINO 추론 + (선택적) NMS + YOLO 라벨 저장 + 이미지 복사 + 시각화
#
# USE_NMS = True  -> NMS 적용 (겹치는 박스 중 신뢰도 높은 것만 유지)
# USE_NMS = False -> NMS 스킵 (탐지된 박스를 그대로 모두 사용)
#
# 이미지를 dataset/images/train 에 함께 복사합니다.
# 라벨(.txt)과 이미지 모두 저장되어야 섹션 2-D 파인튜닝이 정상 작동합니다.

import numpy as np
from PIL import Image as _PIL
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch as _torch

device_gdino = "cuda" if _torch.cuda.is_available() else "cpu"

# ── 여기만 수정하세요 ──────────────────────────
USE_NMS        = False   # True: NMS 적용 / False: NMS 스킵
NMS_IOU_THR    = 0.5     # USE_NMS=True 일 때만 적용
MAX_BOX_RATIO  = 0.45    # 이미지 대비 bbox 최대 크기 비율 (초과 시 오탐으로 제거)
                          # 체인/레일 전체를 묶는 큰 박스 제거용
# ───────────────────────────────────────────────


def nms_boxes(boxes, scores, iou_threshold=0.5):
    # 중복 박스 제거: IoU가 임계값 이상이면 점수 낮은 것 제거
    if len(boxes) == 0:
        return []
    boxes_arr  = np.array(boxes,  dtype=np.float32)
    scores_arr = np.array(scores, dtype=np.float32)
    order = scores_arr.argsort()[::-1]
    keep  = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        xx1 = np.maximum(boxes_arr[i,0], boxes_arr[rest,0])
        yy1 = np.maximum(boxes_arr[i,1], boxes_arr[rest,1])
        xx2 = np.minimum(boxes_arr[i,2], boxes_arr[rest,2])
        yy2 = np.minimum(boxes_arr[i,3], boxes_arr[rest,3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        area_i    = (boxes_arr[i,2]-boxes_arr[i,0]) * (boxes_arr[i,3]-boxes_arr[i,1])
        area_rest = ((boxes_arr[rest,2]-boxes_arr[rest,0]) *
                     (boxes_arr[rest,3]-boxes_arr[rest,1]))
        iou   = inter / (area_i + area_rest - inter + 1e-6)
        order = rest[iou < iou_threshold]
    return keep


def apply_nms_or_passthrough(raw_boxes, raw_scores, raw_cls_ids,
                              use_nms=True, iou_threshold=0.5):
    # use_nms=True  -> 클래스별 NMS 적용 후 유효 인덱스 반환
    # use_nms=False -> NMS 없이 전체 인덱스 그대로 반환
    if not use_nms:
        return list(range(len(raw_boxes)))
    kept_indices = []
    for cid in set(raw_cls_ids):
        idx  = [i for i, c in enumerate(raw_cls_ids) if c == cid]
        kept = nms_boxes(
            [raw_boxes[i]  for i in idx],
            [raw_scores[i] for i in idx],
            iou_threshold=iou_threshold
        )
        kept_indices.extend([idx[k] for k in kept])
    return kept_indices


def run_gdino(img_path: str):
    # 단일 이미지 추론 -> (선택적) NMS -> 크기 필터 -> YOLO 라벨 저장
    # 반환: [(cls_id, cx, cy, bw, bh, score, box_px), ...]
    pil_img = _PIL.open(img_path).convert("RGB")
    W, H    = pil_img.size
    text    = GDINO_PROMPT.replace(".", ". ").strip() + "."

    inputs  = gdino_processor(images=pil_img, text=text,
                               return_tensors="pt").to(device_gdino)
    with _torch.no_grad():
        outputs = gdino_model_hf(**inputs)

    res = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold      = GDINO_BOX_THRESHOLD,
        text_threshold = GDINO_TEXT_THRESHOLD,
        target_sizes   = [(H, W)]
    )[0]

    # 클래스 매핑
    raw_boxes, raw_scores, raw_cls_ids = [], [], []
    for score, label, box in zip(res["scores"], res["text_labels"], res["boxes"]):
        phrase = label.lower()
        cls_id = None
        for cls_name, cid in CLASS_MAP.items():
            if cls_name.lower() in phrase:
                cls_id = cid
                break
        if cls_id is None:
            continue
        raw_boxes.append(box.tolist())
        raw_scores.append(float(score))
        raw_cls_ids.append(cls_id)

    # NMS 또는 전체 통과
    kept = apply_nms_or_passthrough(
        raw_boxes, raw_scores, raw_cls_ids,
        use_nms=USE_NMS, iou_threshold=NMS_IOU_THR
    )
    n_before = len(raw_boxes)
    n_after  = len(kept)
    if USE_NMS and n_before != n_after:
        print(f"    NMS: {n_before}개 -> {n_after}개 "
              f"({n_before - n_after}개 제거, IoU thr={NMS_IOU_THR})")

    # 최종 박스 구성 + 크기 필터
    final = []
    n_size_filtered = 0
    for i in kept:
        x1, y1, x2, y2 = raw_boxes[i]
        bw_ratio = (x2-x1) / W
        bh_ratio = (y2-y1) / H

        # 크기 필터: 이미지 대비 너무 큰 박스는 체인/레일 오탐으로 간주
        if bw_ratio > MAX_BOX_RATIO or bh_ratio > MAX_BOX_RATIO:
            n_size_filtered += 1
            continue

        cx = ((x1+x2)/2) / W
        cy = ((y1+y2)/2) / H
        final.append((raw_cls_ids[i], cx, cy, bw_ratio, bh_ratio,
                      raw_scores[i], raw_boxes[i]))

    if n_size_filtered > 0:
        print(f"    크기 필터: {n_size_filtered}개 제거 "
              f"(bbox 비율 > {MAX_BOX_RATIO}  체인/레일 오탐 의심)")

    return final, pil_img


# ── 저장 폴더 준비 ──
os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)

CLASS_COLOR  = ["#EF9F27", "#185FA5", "#A32D2D"]
CLASS_NAMES  = {v: k for k, v in CLASS_MAP.items()}
nms_label    = f"NMS ON (IoU={NMS_IOU_THR})" if USE_NMS else "NMS OFF"
label_counts = {k: 0 for k in CLASS_MAP}
total_boxes  = 0
img_copied   = 0

# 시각화 그리드 설정
n_imgs = len(image_paths)
n_cols = min(3, n_imgs)
n_rows = (n_imgs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 5, n_rows * 4))
axes = np.array(axes).flatten()

for img_idx, img_path in enumerate(image_paths):
    results, pil_img = run_gdino(img_path)

    # ① 이미지를 dataset/images/train 에 복사
    dst_img = f"dataset/images/train/{Path(img_path).name}"
    if not os.path.exists(dst_img):
        shutil.copy(img_path, dst_img)
        img_copied += 1

    # ② 라벨 .txt 저장
    label_path = f"dataset/labels/train/{Path(img_path).stem}.txt"
    lines = []
    for cls_id, cx, cy, bw, bh, score, box_px in results:
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        label_counts[CLASS_NAMES[cls_id]] += 1
        total_boxes += 1

    with open(label_path, "w") as lf:
        lf.write("\n".join(lines))

    # ③ 시각화
    ax = axes[img_idx]
    ax.imshow(pil_img)

    for cls_id, cx, cy, bw, bh, score, box_px in results:
        x1, y1, x2, y2 = box_px
        color    = CLASS_COLOR[cls_id % len(CLASS_COLOR)]
        cls_name = CLASS_NAMES[cls_id]
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor="none"
        ))
        ax.text(
            x1, max(y1-5, 10),
            f"{cls_name} {score:.2f}",
            color="white", fontsize=8, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85)
        )

    n_boxes  = len(results)
    fname    = Path(img_path).name
    is_new   = not os.path.exists(dst_img)
    ax.set_title(f"{fname}\n{n_boxes} box(es)  [{nms_label}]", fontsize=9)
    ax.axis("off")
    print(f"  {fname}: 박스 {n_boxes}개")

for ax in axes[n_imgs:]:
    ax.axis("off")

plt.suptitle(
    f"Grounding DINO inference result  [{nms_label}]\n"
    f"prompt='{GDINO_PROMPT}'  "
    f"box={GDINO_BOX_THRESHOLD}  text={GDINO_TEXT_THRESHOLD}  "
    f"max_box_ratio={MAX_BOX_RATIO}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

# 최종 요약
train_imgs   = len(glob.glob("dataset/images/train/*.jpg")) + \
               len(glob.glob("dataset/images/train/*.png"))
train_labels = len(glob.glob("dataset/labels/train/*.txt"))

print()
print("=" * 50)
print(f"라벨링 완료: 총 {total_boxes}개 박스  [{nms_label}]")
for k, v in label_counts.items():
    print(f"  {k}: {v}개")
print()
print(f"dataset/images/train  이미지: {train_imgs}장  (이번에 복사: {img_copied}장)")
print(f"dataset/labels/train  라벨:   {train_labels}개")
if train_imgs != train_labels:
    print(f"  이미지({train_imgs})와 라벨({train_labels}) 수가 다릅니다. 확인 필요!")
else:
    print(f"  이미지와 라벨 수 일치 — 섹션 2-C 라벨 검수 후 섹션 2-D 진행 가능")
print("=" * 50)


In [ ]:
# 라벨 파일 상태 확인 (수정본)
# dataset/labels/train/ 기준으로 올바르게 경로를 참조합니다.
import os
from pathlib import Path

def get_label_path(img_path):
    # 이미지 경로에 관계없이 항상 올바른 라벨 경로 반환
    return f"dataset/labels/train/{Path(img_path).stem}.txt"

print("=== 라벨 파일 확인 ===")
found   = 0
missing = 0

for img_path in image_paths:
    fname      = Path(img_path).name
    label_path = get_label_path(img_path)
    label_exists = os.path.exists(label_path)

    if label_exists:
        content = open(label_path).read().strip()
        n_lines = len(content.splitlines()) if content else 0
        print(f"  {fname}")
        print(f"    라벨 경로: {label_path}")
        print(f"    라벨 내용: {n_lines}줄  ->  '{content[:80]}'")
        found += 1
    else:
        print(f"  {fname}")
        print(f"    라벨 경로: {label_path}")
        print(f"    라벨 파일 없음")
        missing += 1
    print()

print("=" * 40)
print(f"라벨 있음: {found}개  /  없음: {missing}개")
if missing == 0:
    print("모든 이미지에 라벨이 존재합니다. 섹션 2-C 검수 후 섹션 2-D 진행 가능합니다.")
else:
    print(f"라벨 없는 이미지 {missing}개 — 섹션 2-B를 다시 실행하세요.")


---
# 섹션 2-C. 라벨 검수

자동 생성된 라벨이 맞는지 시각적으로 확인합니다.
잘못된 박스는 [makesense.ai](https://www.makesense.ai) 또는 labelImg로 수정하세요.


In [ ]:
# 섹션 2-C — 라벨 검수 시각화 (수정본)
# dataset/labels/train/ 기준으로 올바르게 라벨 경로를 참조합니다.
# 차트 제목은 영문으로 표기합니다 (한글 깨짐 방지).

CLASS_COLORS = {0:'#EF9F27', 1:'#185FA5'}
CLASS_NAMES  = {v:k for k,v in CLASS_MAP.items()}

def get_label_path(img_path):
    # 이미지 경로에 관계없이 항상 올바른 라벨 경로 반환
    return f"dataset/labels/train/{Path(img_path).stem}.txt"

check_paths = image_paths[::max(1,len(image_paths)//12)][:12]
n_cols = 3
n_rows = (len(check_paths)+n_cols-1)//n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
axes = np.array(axes).flatten()

for ax, img_path in zip(axes, check_paths):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h_px, w_px = img.shape[:2]
    ax.imshow(img)
    label_path = get_label_path(img_path)
    n_boxes = 0
    if os.path.exists(label_path):
        for line in open(label_path).read().strip().splitlines():
            if not line.strip(): continue
            cls_id = int(line.split()[0])
            cx,cy,bw,bh = [float(x) for x in line.split()[1:5]]
            x1=(cx-bw/2)*w_px; y1=(cy-bh/2)*h_px
            color = CLASS_COLORS.get(cls_id,'red')
            ax.add_patch(patches.Rectangle((x1,y1),bw*w_px,bh*h_px,
                                            lw=2,ec=color,fc='none'))
            ax.text(x1,y1-4,CLASS_NAMES.get(cls_id,f'cls{cls_id}'),
                    color=color,fontsize=9,fontweight='bold')
            n_boxes+=1
    ax.set_title(f'{Path(img_path).name}  |  {n_boxes} box(es)',fontsize=9)
    ax.axis('off')

for ax in axes[len(check_paths):]: ax.axis('off')
plt.suptitle('[Section 2-C] Label Review  |  Orange=gear  Blue=hook',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

# 통계
total_boxes, class_cnts = 0, {k:0 for k in CLASS_MAP}
for img_path in image_paths:
    lp = get_label_path(img_path)
    if not os.path.exists(lp): continue
    for line in open(lp).read().strip().splitlines():
        if not line.strip(): continue
        cls_id = int(line.split()[0])
        for k,v in CLASS_MAP.items():
            if v==cls_id: class_cnts[k]+=1
        total_boxes+=1

print(f"이미지: {len(image_paths)}장  총 박스: {total_boxes}개")
for k,v in class_cnts.items(): print(f"  {k}: {v}개")
if len(image_paths) < 30:
    print(f"\n이미지 {len(image_paths)}장 — 30장 이상 권장")
print()
print('확인:  bbox가 gear/hook을 정확히 감쌈  /  클래스 맞음  /  오탐 없음')


---
# 섹션 2-D. YOLOv8 파인튜닝

## 하이퍼파라미터 가이드

| 파라미터 | 기본값 | 조정 기준 |
|----------|--------|----------|
| `EPOCHS` | 100 | 이미지 30장↓ → 50, 100장↑ → 100 |
| `FREEZE` | 10 | 유지 권장 (COCO 특징 보존) |
| `LR`     | 0.001 | 파인튜닝은 낮은 값 권장 |
| `BATCH`  | 8 | GPU 메모리 부족 시 4 |

## 실행 순서

1. **이 섹션 셀들을 순서대로 실행**하세요.
2. 파인튜닝 완료 후 바로 아래 셀에서 자동으로 새 모델로 검출이 실행됩니다.
3. **섹션 2로 돌아가 설정을 바꿀 필요가 없습니다.**
   섹션 2의 `YOLO_MODE = 'pretrained'` 설정은 항상 초기 상태로 유지합니다.

> 처음부터 다시 실행할 때도 섹션 2 설정값은 그대로이므로 혼란이 없습니다.


In [ ]:
# 섹션 2-D ① — 학습 데이터 준비 및 파인튜닝 실행
#
# 섹션 2의 YOLO_MODE 설정은 건드리지 않습니다.
# 파인튜닝 결과(best.pt)는 아래 셀에서 자동으로 불러와 바로 검출에 사용합니다.
#
# ── 학습 구성 요소 안내 ────────────────────────────────────────
#
#  lr0 (Learning Rate, 학습률)
#    가중치를 한 번에 얼마나 크게 업데이트할지 결정합니다.
#    너무 크면 발산, 너무 작으면 수렴이 느립니다.
#    파인튜닝은 사전학습 가중치를 유지해야 하므로 낮게 설정합니다 (0.001).
#
#  optimizer (옵티마이저)
#    손실값을 줄이는 방향으로 가중치를 실제로 업데이트하는 알고리즘입니다.
#    'auto' 설정 시 YOLOv8이 학습 단계에 맞게 자동으로 선택합니다.
#    직접 지정: 'SGD' / 'Adam' / 'AdamW'
#
#  freeze (레이어 동결)
#    앞 N개 레이어의 가중치를 고정합니다.
#    COCO로 학습된 특징(엣지, 텍스처 등)을 보존하면서
#    뒷부분 레이어만 gear/hook에 맞게 파인튜닝합니다.
#
#  손실 함수 (Loss) — YOLOv8 내부에 세 가지가 합산됩니다.
#    box loss: bbox 좌표 정확도
#    cls loss: 클래스(gear/hook) 분류 정확도
#    dfl loss: bbox 경계 분포의 정밀도
#    학습이 잘 될수록 세 값 모두 점점 작아져야 합니다.
#
#  patience (조기 종료)
#    N 에포크 동안 mAP 개선이 없으면 자동으로 학습을 중단합니다.
#    과적합 방지 및 불필요한 학습 시간 절약에 사용합니다.
#
# ── 시각화 라이브러리 ──────────────────────────────────────────
#
#  USE_WANDB = False (기본)
#    matplotlib으로 학습 완료 후 손실/mAP 곡선을 바로 표시합니다.
#    별도 계정 없이 Colab에서 즉시 사용 가능합니다.
#
#  USE_WANDB = True
#    W&B(Weights & Biases) 웹 대시보드에 실시간으로 지표를 전송합니다.
#    에포크별 loss/mAP 추이, 시스템 자원, 이미지 샘플을 온라인에서 확인 가능합니다.
#    사전 준비: https://wandb.ai 계정 생성 후 API 키 준비 필요.
#
#  TensorBoard (항상 활성화)
#    YOLOv8이 학습 중 자동으로 로그를 생성합니다.
#    아래 셀 실행 후 Colab에서 확인:
#      %load_ext tensorboard
#      %tensorboard --logdir runs/finetune

# ── 여기만 수정하세요 ──────────────────────────
EPOCHS      = 100
BATCH       = 8
IMGSZ       = 640
LR          = 0.001     # 학습률 — 파인튜닝은 낮게 유지 권장
OPTIMIZER   = 'auto'    # 옵티마이저: 'auto' / 'Adam' / 'AdamW' / 'SGD'
FREEZE      = 10        # 동결 레이어 수 — COCO 특징 보존
BASE_MODEL  = 'yolov8s.pt'
RANDOM_SEED = 42        # train/val 분할 재현성 고정
USE_WANDB   = False     # True: W&B 대시보드 사용 (계정 필요)
WANDB_PROJECT = 'coating-line-yolo'  # W&B 프로젝트 이름 (USE_WANDB=True 시 사용)
# ───────────────────────────────────────────────

from ultralytics import YOLO as _YOLO2
import torch as _torch2

# ── W&B 초기화 (USE_WANDB=True 시) ──
if USE_WANDB:
    try:
        import wandb
        wandb.login()
        wandb.init(
            project=WANDB_PROJECT,
            name=f"finetune-ep{EPOCHS}-lr{LR}",
            config={
                "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ,
                "lr0": LR, "optimizer": OPTIMIZER,
                "freeze": FREEZE, "base_model": BASE_MODEL,
            }
        )
        print(f"W&B 연결 완료 → 프로젝트: {WANDB_PROJECT}")
        print(f"  대시보드: https://wandb.ai")
    except ImportError:
        print("W&B 미설치 — 설치 중...")
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"],
                       check=True)
        import wandb
        wandb.login()
        wandb.init(project=WANDB_PROJECT,
                   name=f"finetune-ep{EPOCHS}-lr{LR}")
        print("W&B 설치 및 연결 완료")
else:
    print("시각화: matplotlib (학습 완료 후 손실/mAP 곡선 표시)")
    print("TensorBoard 로그도 자동 생성됩니다.")
    print("  Colab에서 확인하려면 새 셀에서 실행:")
    print("  %load_ext tensorboard")
    print("  %tensorboard --logdir runs/finetune")

# ── 학습 데이터 존재 여부 사전 확인 ──
all_imgs = (glob.glob("dataset/images/train/*.jpg") +
            glob.glob("dataset/images/train/*.png"))

if len(all_imgs) == 0:
    print("=" * 55)
    print("dataset/images/train 에 이미지가 없습니다.")
    print()
    print("파인튜닝 전에 아래 순서를 완료했는지 확인하세요:")
    print("  1) 섹션 2-B: Grounding DINO 라벨링")
    print("     -> dataset/images/train/ 에 이미지+라벨 저장")
    print("  2) 섹션 2-C: 라벨 검수")
    print()
    print("현재 dataset 폴더 상태:")
    for root, dirs, files in os.walk("dataset"):
        depth = root.replace("dataset", "").count(os.sep)
        print(f"{'  '*depth}{os.path.basename(root)}/  ({len(files)}개)")
    print("=" * 55)
    raise RuntimeError("이미지 없음 — 섹션 2-B 먼저 실행하세요.")

if len(all_imgs) < 10:
    print(f"이미지 {len(all_imgs)}장 — 30장 이상 권장. 적을수록 과적합 위험 높음.")

# ── train/val 분할 (고정 시드) ──
random.seed(RANDOM_SEED)
random.shuffle(all_imgs)
val_n    = max(1, int(len(all_imgs) * 0.2))

# val 분할 후 train이 최소 1장은 남아야 함
if len(all_imgs) - val_n < 1:
    val_n = 0
    print("이미지가 너무 적어 val 분할을 건너뜁니다. train 전체로 학습합니다.")

val_imgs = all_imgs[:val_n]
os.makedirs("dataset/images/val", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)
for img_p in val_imgs:
    lbl_p = f"dataset/labels/train/{Path(img_p).stem}.txt"
    shutil.move(img_p, f"dataset/images/val/{Path(img_p).name}")
    if os.path.exists(lbl_p):
        shutil.move(lbl_p, f"dataset/labels/val/{Path(img_p).stem}.txt")

print(f"데이터 분할  train:{len(all_imgs)-val_n}장  val:{val_n}장  (seed={RANDOM_SEED})")

# ── data.yaml 생성 ──
data_yaml_dict = {
    "path": "/content/dataset",
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_MAP),
    "names": {v: k for k, v in CLASS_MAP.items()},
}
with open("data.yaml", "w") as yf:
    yaml.dump(data_yaml_dict, yf, allow_unicode=True)
print(f"data.yaml 생성  클래스: {data_yaml_dict['names']}")

# ── 파인튜닝 실행 ──
print(f"\n파인튜닝 시작")
print(f"  베이스 모델 : {BASE_MODEL}")
print(f"  에포크      : {EPOCHS}  (patience={15}으로 조기 종료 가능)")
print(f"  배치        : {BATCH}")
print(f"  학습률(lr0) : {LR}")
print(f"  옵티마이저  : {OPTIMIZER}")
print(f"  동결 레이어 : {FREEZE}개 (COCO 특징 보존)")
print(f"  GPU         : {'사용 가능' if _torch2.cuda.is_available() else '없음 (CPU 학습 — 매우 느림)'}")
print()

ft_model = _YOLO2(BASE_MODEL)
ft_model.train(
    data="data.yaml",
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=LR,
    optimizer=OPTIMIZER,
    freeze=FREEZE,
    patience=15,
    augment=True,
    degrees=10,
    fliplr=0.5,
    mosaic=0.8,
    project="runs/finetune",
    name="coating_line",
    exist_ok=True,
    verbose=False,
)

FINETUNED_PT = "runs/finetune/coating_line/weights/best.pt"
print(f"\n파인튜닝 완료 -> {FINETUNED_PT}")

# ── W&B 종료 ──
if USE_WANDB:
    wandb.finish()
    print("W&B 로그 업로드 완료")

print("다음 셀을 실행하면 손실/mAP 곡선 확인 후 새 모델로 바로 검출합니다.")


In [ ]:
# 섹션 2-D ① 보조 — 학습 곡선 시각화
#
# 파인튜닝 완료 후 실행하세요.
# YOLOv8이 저장한 results.csv 를 읽어 아래 4가지를 한눈에 표시합니다.
#
#  [좌상] Train Loss  — 학습 손실 (box+cls+dfl 합산): 낮을수록 좋음
#  [우상] Val Loss    — 검증 손실: train loss와 함께 내려가야 정상
#                       val loss만 올라가면 과적합 신호
#  [좌하] mAP@50      — IoU 0.5 기준 평균 정밀도: 높을수록 좋음 (0.70+ 권장)
#  [우하] mAP@50-95   — IoU 0.5~0.95 엄격 기준: mAP@50보다 낮은 게 정상

import pandas as pd
import matplotlib.pyplot as plt
import glob as _gl
import os

# results.csv 경로 탐색
csv_candidates = _gl.glob("runs/finetune/coating_line/results.csv")
if not csv_candidates:
    csv_candidates = _gl.glob("**/results.csv", recursive=True)

if not csv_candidates:
    print("results.csv 를 찾을 수 없습니다.")
    print("섹션 2-D ① 파인튜닝 셀을 먼저 실행하세요.")
else:
    csv_path = csv_candidates[0]
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # 공백 제거
    print(f"학습 로그 로드: {csv_path}")
    print(f"  총 {len(df)} 에포크 기록")
    print(f"  컬럼: {list(df.columns)}")
    print()

    # 컬럼명 매핑 (YOLOv8 버전마다 약간씩 다름)
    def find_col(df, keywords):
        for col in df.columns:
            if all(k.lower() in col.lower() for k in keywords):
                return col
        return None

    col_train_box = find_col(df, ['train', 'box'])
    col_train_cls = find_col(df, ['train', 'cls'])
    col_val_box   = find_col(df, ['val',   'box'])
    col_val_cls   = find_col(df, ['val',   'cls'])
    col_map50     = find_col(df, ['map50']) if find_col(df, ['map50']) and 'map50-95' not in find_col(df, ['map50']) else None
    col_map50     = find_col(df, ['metrics/mAP50(B)']) or find_col(df, ['map50'])
    col_map5095   = find_col(df, ['metrics/mAP50-95(B)']) or find_col(df, ['map50-95'])

    epochs = df['epoch'] if 'epoch' in df.columns else range(len(df))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # [좌상] Train Loss
    ax = axes[0][0]
    if col_train_box:
        ax.plot(epochs, df[col_train_box], label='box loss', color='#EF9F27', lw=1.5)
    if col_train_cls:
        ax.plot(epochs, df[col_train_cls], label='cls loss', color='#185FA5', lw=1.5)
    if col_train_box and col_train_cls:
        ax.plot(epochs, df[col_train_box] + df[col_train_cls],
                label='total', color='#A32D2D', lw=2, linestyle='--')
    ax.set_title('Train Loss  (lower = better)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # [우상] Val Loss
    ax = axes[0][1]
    if col_val_box:
        ax.plot(epochs, df[col_val_box], label='val box loss', color='#EF9F27', lw=1.5)
    if col_val_cls:
        ax.plot(epochs, df[col_val_cls], label='val cls loss', color='#185FA5', lw=1.5)
    ax.set_title('Validation Loss  (should decrease with train loss)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # [좌하] mAP@50
    ax = axes[1][0]
    if col_map50:
        ax.plot(epochs, df[col_map50], color='#0F6E56', lw=2, label='mAP@50')
        ax.axhline(0.70, color='gray', linestyle='--', lw=1, label='0.70 target')
        best_map50 = df[col_map50].max()
        best_ep    = df[col_map50].idxmax()
        ax.scatter([epochs.iloc[best_ep] if hasattr(epochs, 'iloc') else best_ep],
                   [best_map50], color='red', zorder=5, s=60)
        ax.annotate(f"best: {best_map50:.3f}",
                    xy=(epochs.iloc[best_ep] if hasattr(epochs, 'iloc') else best_ep, best_map50),
                    xytext=(10, -15), textcoords='offset points', fontsize=9, color='red')
    ax.set_title('mAP@50  (target >= 0.70)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('mAP@50')
    ax.set_ylim(0, 1.05); ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # [우하] mAP@50-95
    ax = axes[1][1]
    if col_map5095:
        ax.plot(epochs, df[col_map5095], color='#534AB7', lw=2, label='mAP@50-95')
        best_map5095 = df[col_map5095].max()
        best_ep2     = df[col_map5095].idxmax()
        ax.scatter([epochs.iloc[best_ep2] if hasattr(epochs, 'iloc') else best_ep2],
                   [best_map5095], color='red', zorder=5, s=60)
        ax.annotate(f"best: {best_map5095:.3f}",
                    xy=(epochs.iloc[best_ep2] if hasattr(epochs, 'iloc') else best_ep2, best_map5095),
                    xytext=(10, -15), textcoords='offset points', fontsize=9, color='red')
    ax.set_title('mAP@50-95  (stricter metric)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('mAP@50-95')
    ax.set_ylim(0, 1.05); ax.legend(fontsize=9); ax.grid(alpha=0.3)

    plt.suptitle(
        f'[Section 2-D] Training Curves  |  '
        f'model={BASE_MODEL}  epochs={len(df)}  lr={LR}  optimizer={OPTIMIZER}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    # 판정 요약
    print("=" * 50)
    if col_map50:
        print(f"  최고 mAP@50    : {best_map50:.3f}  (에포크 {best_ep})")
    if col_map5095:
        print(f"  최고 mAP@50-95 : {best_map5095:.3f}")
    print()

    if col_map50:
        if best_map50 >= 0.70:
            print("판정: GOOD — 섹션 2-D ② 검출 확인 후 섹션 2-E로 진행하세요.")
        elif best_map50 >= 0.50:
            print("판정: FAIR — 사용 가능하나 이미지 추가 후 재학습을 권장합니다.")
        else:
            print("판정: LOW  — 학습 이미지 부족 또는 라벨 오류 가능성 있음.")
            print("  섹션 2-B로 돌아가 이미지 추가 및 재라벨링 후 다시 실행하세요.")

    if col_train_box and col_val_box:
        last_train = (df[col_train_box] + df[col_train_cls]).iloc[-1]
        last_val   = (df[col_val_box]   + df[col_val_cls]).iloc[-1]
        if last_val > last_train * 1.5:
            print()
            print("과적합 의심: val loss 가 train loss 보다 1.5배 이상 높습니다.")
            print("  이미지를 추가하거나 EPOCHS를 줄여보세요.")
    print("=" * 50)


In [ ]:
# 섹션 2-D ② — 파인튜닝 모델로 즉시 검출 확인
#
# 섹션 2로 돌아가지 않아도 됩니다.
# FINETUNED_PT 경로를 사용해 바로 검출을 실행합니다.
# 섹션 2의 YOLO_MODE = 'pretrained' 설정은 그대로 유지됩니다.

import glob as _g2d
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2, numpy as np
from ultralytics import YOLO as _YOLO_ft
from pathlib import Path

# ── best.pt 경로 확인 ──
if 'FINETUNED_PT' not in dir() or not os.path.exists(FINETUNED_PT):
    candidates = _g2d.glob("**/best.pt", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            "best.pt 없음 — 섹션 2-D ① 파인튜닝 셀을 먼저 실행하세요.")
    FINETUNED_PT = max(candidates, key=os.path.getmtime)

size_mb = os.path.getsize(FINETUNED_PT) / 1024 / 1024
print(f"파인튜닝 모델 로드: {FINETUNED_PT}  ({size_mb:.1f} MB)")

ft_detect_model = _YOLO_ft(FINETUNED_PT)
print(f"클래스: {ft_detect_model.names}")
print()

# ── 테스트 이미지 선택 ──
# builtin 모드: BUILTIN_SAVE_DIR / 실제 영상 모드: FRAMES에서 샘플 저장 후 사용
_test_src = sorted(_g2d.glob(f"{BUILTIN_SAVE_DIR}/*.jpg"))
if not _test_src:
    _test_src = sorted(_g2d.glob("dataset/images/**/*.jpg", recursive=True))

CONF_THR    = 0.45   # 검출 신뢰도 임계값 (파인튜닝 직후라면 0.40~0.50 권장)
N_SHOW      = 12     # 표시할 이미지 수
step        = max(1, len(_test_src) // N_SHOW)
test_images = _test_src[::step][:N_SHOW]
print(f"테스트 이미지: {len(test_images)}장  (conf_thr={CONF_THR})")
print()

if not test_images:
    print("테스트할 이미지가 없습니다.")
    print(f"  builtin 모드: {BUILTIN_SAVE_DIR} 폴더를 확인하세요.")
else:
    n_cols = 3
    n_rows = (len(test_images) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 5, n_rows * 4))
    axes = np.array(axes).flatten()

    detected_count = missed_count = 0

    for ax, img_path in zip(axes, test_images):
        frame = cv2.imread(img_path)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = ft_detect_model(frame, conf=CONF_THR, verbose=False)
        boxes   = results[0].boxes

        ax.imshow(rgb)
        n_boxes = 0
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            conf     = float(box.conf)
            cls_id   = int(box.cls)
            cls_name = ft_detect_model.names[cls_id]
            color    = '#EF9F27' if cls_id == 0 else '#185FA5'
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none'
            ))
            ax.text(x1, max(y1-5, 10),
                    f"{cls_name} {conf:.2f}",
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor=color, alpha=0.85))
            n_boxes += 1

        fname = Path(img_path).name
        title_color = 'green' if n_boxes > 0 else 'red'
        ax.set_title(f"{fname}\n{n_boxes} detection(s)",
                     fontsize=9, color=title_color)
        ax.axis('off')
        (detected_count if n_boxes > 0 else missed_count).__class__  # dummy
        if n_boxes > 0:
            detected_count += 1
        else:
            missed_count += 1

    for ax in axes[len(test_images):]:
        ax.axis('off')

    plt.suptitle(
        f"[Section 2-D] Fine-tuned model detection  |  "
        f"model={Path(FINETUNED_PT).parent.parent.name}\n"
        f"detected={detected_count}/{len(test_images)}  "
        f"missed={missed_count}/{len(test_images)}  "
        f"conf_thr={CONF_THR}",
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    # ── 결과 요약 ──
    rate = detected_count / len(test_images)
    print(f"탐지 성공: {detected_count}/{len(test_images)}장  ({rate*100:.0f}%)")
    print(f"탐지 실패: {missed_count}/{len(test_images)}장")
    print()

    if rate >= 0.8:
        print("결과: 양호")
        print(f"  → 이후 섹션은 이 모델을 자동으로 사용합니다: {FINETUNED_PT}")
        print("  → 섹션 2-E(성능 검증)를 이어서 실행하세요.")
    elif rate >= 0.5:
        print("결과: 보통 — 이미지를 더 추가하고 파인튜닝을 다시 권장합니다.")
    else:
        print("결과: 미흡 — 학습 이미지 부족 또는 라벨 오류 가능성이 있습니다.")
        print("  섹션 2-B로 돌아가 이미지 추가 후 재라벨링하세요.")

    print()
    print("※ 섹션 2의 YOLO_MODE 설정은 변경되지 않았습니다.")
    print(f"  파인튜닝 모델 경로: {FINETUNED_PT}")
    print("  이 경로는 FINETUNED_PT 변수로 이후 섹션에서도 참조 가능합니다.")


---
# 섹션 2-E. 성능 검증

## 판정 기준

| mAP@50 | 판정 | 조치 |
|--------|------|------|
| 0.70↑ | GOOD | 섹션 2 custom 모드로 교체 |
| 0.50~0.70 | FAIR | 사용 가능, 이미지 추가 권장 |
| 0.50↓ | LOW | 이미지 추가 후 재실행 |


In [ ]:
import glob as _glob
from ultralytics import YOLO as _YOLO3

candidates = _glob.glob("**/best.pt", recursive=True)
if not candidates:
    raise FileNotFoundError("best.pt 없음 — 섹션 2-D를 먼저 실행하세요.")

BEST_PT   = max(candidates, key=os.path.getmtime)
size_mb   = os.path.getsize(BEST_PT)/1024/1024
print(f"검증 모델: {BEST_PT}  ({size_mb:.1f}MB)")

val_model = _YOLO3(BEST_PT)
metrics   = val_model.val(data="data.yaml", verbose=False)
map50     = metrics.box.map50

print("="*50)
print(f"  mAP@50    : {map50:.3f}  (0.70 이상 권장)")
print(f"  mAP@50-95 : {metrics.box.map:.3f}")
print(f"  Precision : {metrics.box.mp:.3f}")
print(f"  Recall    : {metrics.box.mr:.3f}")
print("="*50)

if   map50 >= 0.70: print("판정: GOOD  → 섹션 2 custom 모드로 교체")
elif map50 >= 0.50: print("판정: FAIR  → 사용 가능, 이미지 추가 권장")
else:               print("판정: LOW   → 섹션 2-B로 돌아가 이미지 추가")

print()
print("섹션 2에 적용할 값:")
print(f"  YOLO_MODE    = 'custom'")
print(f"  YOLO_WEIGHTS = '{BEST_PT}'")
print(f"  YOLO_CLASSES = [0, 1]")


In [ ]:
# 파인튜닝 모델 테스트 — 실제 이미지에서 gear 탐지 확인

import glob, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np
from ultralytics import YOLO as _YOLO_test
from pathlib import Path

# ── 테스트할 모델 로드 ──
candidates = glob.glob("**/best.pt", recursive=True)
if not candidates:
    print("best.pt 없음 — 파인튜닝을 먼저 실행하세요.")
else:
    BEST_PT    = max(candidates, key=os.path.getmtime)
    test_model = _YOLO_test(BEST_PT)
    print(f"모델: {BEST_PT}")
    print(f"클래스: {test_model.names}")
    print()

    # ── 테스트할 이미지 선택 ──
    # builtin_frames 전체 또는 일부 선택
    test_images = sorted(glob.glob("/content/builtin_frames/*.jpg"))
    # 균등 간격으로 12장 선택
    step        = max(1, len(test_images) // 12)
    test_images = test_images[::step][:12]
    print(f"테스트 이미지: {len(test_images)}장")
    print()

    # ── 추론 + 시각화 ──
    n_cols   = 3
    n_rows   = (len(test_images) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 5, n_rows * 4))
    axes = np.array(axes).flatten()

    detected_count = 0
    missed_count   = 0

    for ax, img_path in zip(axes, test_images):
        frame = cv2.imread(img_path)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = test_model(frame, conf=0.65, verbose=False)
        boxes   = results[0].boxes

        ax.imshow(rgb)

        n_boxes = 0
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            conf     = float(box.conf)
            cls_id   = int(box.cls)
            cls_name = test_model.names[cls_id]

            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='#EF9F27', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(
                x1, max(y1-5, 10),
                f"{cls_name} {conf:.2f}",
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',
                           facecolor='#EF9F27', alpha=0.85)
            )
            n_boxes += 1

        fname = Path(img_path).name
        color = 'green' if n_boxes > 0 else 'red'
        ax.set_title(f"{fname}\n{n_boxes} gear detected",
                     fontsize=9, color=color)
        ax.axis('off')

        if n_boxes > 0:
            detected_count += 1
        else:
            missed_count += 1

    for ax in axes[len(test_images):]:
        ax.axis('off')

    plt.suptitle(
        f"Fine-tuned model test  |  model={Path(BEST_PT).parent.parent.name}\n"
        f"detected={detected_count}/{len(test_images)}  "
        f"missed={missed_count}/{len(test_images)}  "
        f"conf_thr=0.65",
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    # ── 요약 ──
    print(f"탐지 성공: {detected_count}/{len(test_images)}장  "
          f"({detected_count/len(test_images)*100:.0f}%)")
    print(f"탐지 실패: {missed_count}/{len(test_images)}장")
    print()
    if detected_count / len(test_images) >= 0.8:
        print("결과: 양호 — 섹션 2에서 YOLO_MODE=custom 으로 계속 진행")
    elif detected_count / len(test_images) >= 0.5:
        print("결과: 보통 — 이미지를 더 추가하고 파인튜닝을 다시 권장")
    else:
        print("결과: 미흡 — 파인튜닝이 제대로 안 됐습니다")
        print("  원인 1: 학습 이미지 부족 (현재 2장 → 30장 이상 필요)")
        print("  원인 2: val 라벨 없어서 mAP 측정 불가 상태로 학습됨")
        print("  해결: builtin_frames 50장으로 라벨링 후 재파인튜닝")

---
# 섹션 3. Tracker — 프레임 간 ID 유지

## 역할

YOLO는 매 프레임을 독립 처리하므로 "이 프레임의 박스 = 저 프레임의 박스"를 모릅니다.
GRU가 20프레임 연속으로 같은 톱니를 봐야 하므로 Tracker가 ID를 유지합니다.

## 파라미터 가이드

| 파라미터 | 기본값 | 조정 기준 |
|----------|--------|----------|
| `IOU_THRESH` | 0.25 | 낮추면 컨베이어 빠를 때 유리 |
| `MAX_LOST` | 5 | 높이면 분진으로 가려져도 ID 유지 |
| `MIN_HITS` | 2 | 낮추면 빠르게 추적 시작 |


In [ ]:
# 섹션 3 — 파인튜닝 모델 자동 적용 + IoUTracker 정의
#
# YOLO_MODE 설정은 항상 'pretrained' 초기 상태를 유지합니다.
# 파인튜닝 모델(best.pt)이 존재하면 자동으로 detect_objects 함수를 교체합니다.
# 파인튜닝 모델이 없으면 현재 YOLO_MODE 설정을 그대로 사용합니다.

import glob as _gl3
from ultralytics import YOLO as _YOLO_s3

# ── 파인튜닝 모델 자동 탐지 및 detect_objects 교체 ──
_ft_candidates = _gl3.glob('**/best.pt', recursive=True)

if _ft_candidates:
    _ft_pt = max(_ft_candidates, key=os.path.getmtime)
    _ft_model_s3 = _YOLO_s3(_ft_pt)
    _ft_conf = 0.45  # 파인튜닝 모델 신뢰도 임계값 (필요 시 조정)

    def detect_objects(frame):
        results = _ft_model_s3(frame, conf=_ft_conf, verbose=False)
        return [
            {'bbox': tuple(int(v) for v in box.xyxy[0].tolist()),
             'conf': float(box.conf),
             'cls':  int(box.cls)}
            for box in results[0].boxes
        ]

    print(f'파인튜닝 모델 자동 적용: {_ft_pt}')
    print(f'클래스: {_ft_model_s3.names}  conf_thr={_ft_conf}')
    print('섹션 2의 YOLO_MODE 설정은 변경되지 않았습니다.')

    # 검출 테스트
    _test_dets = detect_objects(FRAMES[0])
    print(f'\n검출 테스트 (Frame 0): {len(_test_dets)}개')
    for d in _test_dets:
        print(f'  cls={d["cls"]}  conf={d["conf"]:.3f}  bbox={d["bbox"]}')
    if not _test_dets:
        print('검출 0개 — conf_thr를 낮추거나 FRAMES 확인이 필요합니다.')
        print(f'  현재 _ft_conf = {_ft_conf}  ->  0.30 으로 낮춰보세요.')
else:
    print(f'파인튜닝 모델 없음 — 현재 설정으로 진행합니다.')
    print(f'  YOLO_MODE = {YOLO_MODE}  /  YOLO_CONF = {YOLO_CONF}')
    print('  섹션 2-D 파인튜닝을 완료하면 다음 실행부터 자동으로 적용됩니다.')

print()

# ── IoUTracker 정의 ──
# 프레임 간 bbox 겹침(IoU)으로 동일 톱니를 연결합니다.
# MIN_HITS 연속 매칭 이후 'confirmed' 상태가 됩니다.

def compute_iou(a,b):
    xa=max(a[0],b[0]); ya=max(a[1],b[1])
    xb=min(a[2],b[2]); yb=min(a[3],b[3])
    inter=max(0,xb-xa)*max(0,yb-ya)
    aa=(a[2]-a[0])*(a[3]-a[1]); ab=(b[2]-b[0])*(b[3]-b[1])
    union=aa+ab-inter
    return inter/union if union>0 else 0.0

@dataclass
class Track:
    track_id:int; bbox:Tuple; age:int=0; lost_count:int=0

class IoUTracker:
    IOU_THRESH=0.25; MAX_LOST=5; MIN_HITS=2

    def __init__(self):
        self.tracks:List[Track]=[]; self._next_id=100

    def update(self, detections:List[Dict]) -> List[Dict]:
        det_bboxes=[d['bbox'] for d in detections]
        matched_t,matched_d=set(),set()
        if self.tracks and det_bboxes:
            iou_mat=np.array([[compute_iou(t.bbox,db) for db in det_bboxes]
                               for t in self.tracks])
            for flat in np.argsort(-iou_mat.ravel()):
                ti,di=divmod(flat,len(det_bboxes))
                if iou_mat[ti,di]<self.IOU_THRESH: break
                if ti in matched_t or di in matched_d: continue
                self.tracks[ti].bbox=det_bboxes[di]
                self.tracks[ti].age+=1; self.tracks[ti].lost_count=0
                matched_t.add(ti); matched_d.add(di)
        for ti,t in enumerate(self.tracks):
            if ti not in matched_t: t.lost_count+=1
        for di,det in enumerate(detections):
            if di not in matched_d:
                self.tracks.append(Track(self._next_id,det['bbox']))
                self._next_id+=1
        self.tracks=[t for t in self.tracks if t.lost_count<=self.MAX_LOST]
        return [{'track_id':t.track_id,'bbox':t.bbox,
                 'confirmed':t.age>=self.MIN_HITS,'age':t.age}
                for t in self.tracks]

print('IoUTracker 정의 완료')
print(f'  IOU_THRESH = {IoUTracker.IOU_THRESH}')
print(f'  MAX_LOST   = {IoUTracker.MAX_LOST}')
print(f'  MIN_HITS   = {IoUTracker.MIN_HITS}')


In [ ]:
# 섹션 3 — Tracker ID 연속성 확인
# 산점도: 프레임별 confirmed Track ID (점이 이어지면 안정적 추적)
# 막대그래프: 프레임별 활성 Track 수
# 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

tracker_test=IoUTracker(); frame_ids,frame_bboxes=[],[]
for frame in FRAMES:
    dets=detect_objects(frame); tracked=tracker_test.update(dets)
    confirmed=[t for t in tracked if t['confirmed']]
    frame_ids.append([t['track_id'] for t in confirmed])
    frame_bboxes.append({t['track_id']:t['bbox'] for t in confirmed})

fig,axes=plt.subplots(2,1,figsize=(14,6))
all_ids=sorted(set(tid for ids in frame_ids for tid in ids))
colors=plt.cm.Set2(np.linspace(0,1,max(len(all_ids),1)))
id_color={tid:colors[i] for i,tid in enumerate(all_ids)}
for fi,ids in enumerate(frame_ids):
    for tid in ids:
        axes[0].scatter(fi,tid,color=id_color.get(tid,'gray'),s=20,zorder=3)
axes[0].set_ylabel('Track ID')
axes[0].set_title('Track ID per Frame  |  Continuous = stable tracking')
if all_ids: axes[0].set_yticks(all_ids)
axes[0].grid(axis='x',alpha=0.3)

track_counts=[len(ids) for ids in frame_ids]
axes[1].bar(range(len(track_counts)),track_counts,color='#185FA5',alpha=0.7)
axes[1].axhline(1,color='#0F6E56',ls='--',alpha=0.6,label='Target: 1 track')
axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Active Tracks')
axes[1].set_title('Confirmed Track Count per Frame'); axes[1].legend()

plt.suptitle('[Section 3] Tracker ID Continuity',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

detected=sum(1 for ids in frame_ids if ids)
print(f'트래킹 프레임: {detected}/{len(FRAMES)}  생성된 ID: {all_ids}')
print()

if detected == 0:
    print('트래킹 0 — detect_objects 가 검출을 못하고 있습니다.')
    print('위 셀(섹션 3 첫 번째 셀)의 검출 테스트 결과를 확인하세요.')
    print('검출 테스트도 0개였다면 _ft_conf 값을 낮춰보세요 (예: 0.30).')
elif detected < len(FRAMES) * 0.5:
    print('트래킹이 자주 끊깁니다. 아래 값을 조정하세요:')
    print(f'  IoUTracker.IOU_THRESH 낮추기 (현재: {IoUTracker.IOU_THRESH})')
    print(f'  IoUTracker.MAX_LOST   높이기  (현재: {IoUTracker.MAX_LOST})')
else:
    print('트래킹 안정적입니다. 섹션 4로 진행하세요.')


---
# 섹션 4. 동적 ROI 생성

## 역할

전체 프레임에서 Flow를 계산하면 배경·레일 등 불필요한 움직임이 섞입니다.
bbox 위치에서 톱니 주변만 잘라내고, 컨베이어 전진 벡터를 빼서
**순수 회전 성분만** GRU에 전달합니다.

- `PADDING`: bbox보다 여유를 줘서 이빨 가장자리 신호 포함
- 이동 성분 보정: `bbox 중심 차분` → Flow에서 차감


In [ ]:
# ── 여기만 수정하세요 ──────────────────────────
PADDING  = 25   # bbox 여유 픽셀 (톱니 크기에 맞게 조정)
ROI_SIZE = 64   # CNN 입력 크기 (고정)
# ───────────────────────────────────────────────

class DynamicROI:
    def __init__(self,padding=PADDING,roi_size=ROI_SIZE):
        self.padding=padding; self.roi_size=roi_size
        self._prev_center:Dict[int,Tuple]={}

    def crop(self,track_id:int,bbox:Tuple,frame:np.ndarray)->Dict:
        H,W=frame.shape[:2]; x1,y1,x2,y2=bbox
        cx=(x1+x2)/2; cy=(y1+y2)/2
        prev=self._prev_center.get(track_id)
        translation=(cx-prev[0],cy-prev[1]) if prev else (0.0,0.0)
        self._prev_center[track_id]=(cx,cy)
        rx1=max(0,int(x1-self.padding)); ry1=max(0,int(y1-self.padding))
        rx2=min(W,int(x2+self.padding)); ry2=min(H,int(y2+self.padding))
        roi=frame[ry1:ry2,rx1:rx2]
        if roi.size==0:
            roi=np.zeros((self.roi_size,self.roi_size),dtype=np.uint8)
        else:
            if len(roi.shape)==3: roi=cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)
            roi=cv2.resize(roi,(self.roi_size,self.roi_size))
        return {'roi':roi,'roi_bbox':(rx1,ry1,rx2,ry2),'translation':translation}

    def flush(self,track_id:int):
        self._prev_center.pop(track_id,None)


In [ ]:
# 섹션 4 — 동적 ROI 추출 및 Track ID별 이동 벡터 시각화
#
# DynamicROI는 Track ID별로 독립적으로 이전 bbox 중심을 기억합니다.
# 따라서 dx/dy는 항상 '동일 톱니'의 프레임 간 이동량입니다.
# 여러 톱니가 동시에 잡혀도 ID가 다르면 서로 영향을 주지 않습니다.
#
# [패널 A] 프레임별 ROI 시각화 (원본 + 크롭)
# [패널 B] Track ID별 수평 이동량(dx) — 선이 겹치지 않고 ID별로 구분
# [패널 C] 전체 dx 평균 — Optical Flow 보정에 실제 사용되는 값
# ※ 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

roi_module  = DynamicROI()
tracker_s4  = IoUTracker()
roi_samples = []

# Track ID별 dx 로그: {track_id: [(frame_idx, dx), ...]}
track_dx_log = {}

for fi, frame in enumerate(FRAMES):
    dets    = detect_objects(frame)
    tracked = tracker_s4.update(dets)
    for t in tracked:
        if not t['confirmed']: continue
        tid = t['track_id']
        res = roi_module.crop(tid, t['bbox'], frame)
        dx  = res['translation'][0]

        # Track ID별 독립 로그
        if tid not in track_dx_log:
            track_dx_log[tid] = []
        track_dx_log[tid].append((fi, dx))

        # ROI 샘플 수집 (시각화용, 4개만)
        if fi % max(1, len(FRAMES)//4) == 0 and len(roi_samples) < 4:
            roi_samples.append((fi, frame, res['roi'],
                                res['roi_bbox'], res['translation'], tid))

# ── 패널 A: ROI 시각화 ──
n_show = len(roi_samples)
if n_show > 0:
    fig, axes = plt.subplots(n_show, 2, figsize=(12, 3.5*n_show))
    if n_show == 1: axes = [axes]
    for row, (fi, frame, roi_img, roi_bbox, trans, tid) in enumerate(roi_samples):
        vis = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB)
        rx1, ry1, rx2, ry2 = roi_bbox
        axes[row][0].imshow(vis)
        axes[row][0].add_patch(patches.Rectangle(
            (rx1,ry1), rx2-rx1, ry2-ry1, lw=2, ec='#EF9F27', fc='none'))
        axes[row][0].set_title(
            f'Frame {fi}  Track ID={tid}  ROI (orange)  dx={trans[0]:.1f}px',
            fontsize=10)
        axes[row][0].axis('off')
        axes[row][1].imshow(roi_img, cmap='gray')
        axes[row][1].set_title(
            f'Cropped ROI  ({roi_img.shape[0]}x{roi_img.shape[1]}px)', fontsize=10)
        axes[row][1].axis('off')
    plt.suptitle(
        f'[Section 4] Dynamic ROI  |  padding={PADDING}px  size={ROI_SIZE}x{ROI_SIZE}',
        fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

# ── 패널 B: Track ID별 수평 이동량(dx) ──
# 각 ID가 독립된 선으로 표시되어 동일 톱니의 이동만 확인 가능
if track_dx_log:
    fig, ax = plt.subplots(figsize=(14, 4))
    colors_map = plt.cm.Set2(np.linspace(0, 1, len(track_dx_log)))

    for (tid, log), color in zip(sorted(track_dx_log.items()), colors_map):
        fis = [x[0] for x in log]
        dxs = [x[1] for x in log]
        ax.plot(fis, dxs, lw=1.2, color=color, label=f'Track ID={tid}',
                alpha=0.85)

    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_ylabel('dx (px)')
    ax.set_xlabel('Frame index')
    ax.set_title(
        'Per-track horizontal translation (dx)  |  '
        'Each line = same gear tracked continuously')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # ── 패널 C: 전체 평균 dx (Flow 보정에 실제 사용되는 값) ──
    # 동시에 여러 Track이 있을 때는 해당 프레임의 dx 평균을 사용
    frame_dx_avg = {}
    for tid, log in track_dx_log.items():
        for fi, dx in log:
            if fi not in frame_dx_avg:
                frame_dx_avg[fi] = []
            frame_dx_avg[fi].append(dx)

    fis_avg = sorted(frame_dx_avg.keys())
    dxs_avg = [np.mean(frame_dx_avg[fi]) for fi in fis_avg]

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(fis_avg, dxs_avg, color='#185FA5', lw=1.5,
            label='mean dx (used for Flow correction)')
    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_ylabel('Translation (px)')
    ax.set_xlabel('Frame index')
    ax.set_title(
        'Conveyor translation vector  |  '
        'subtracted from Flow to isolate gear rotation')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # 요약
    all_dxs = [dx for log in track_dx_log.values() for _, dx in log]
    print(f'Track ID 수: {len(track_dx_log)}개  '
          f'{list(track_dx_log.keys())}')
    print(f'전체 평균 이동: {np.mean(all_dxs):.2f}px/프레임')
    print()
    print('Track ID별 평균 이동:')
    for tid, log in sorted(track_dx_log.items()):
        dxs = [dx for _, dx in log]
        print(f'  ID={tid}: 평균 {np.mean(dxs):.2f}px  '
              f'(관측 {len(dxs)}프레임  '
              f'프레임 {log[0][0]}~{log[-1][0]})')
    print()
    print(f'PADDING={PADDING}px  |  '
          f'톱니가 ROI에 너무 작거나 크면 조정하세요.')


---
# 섹션 5. Optical Flow

## 역할

"톱니가 회전 중인가"를 픽셀 움직임으로 측정합니다.

```
정상 회전 → Flow 벡터 규칙적 (방향·크기 일정)
회전 불량 → Flow 벡터 흐트러짐
이탈 확정 → Flow 벡터 소멸 (크기 0에 수렴)
```

출력: `(H, W, 2)` float32 — 수평속도(u), 수직속도(v)
시각화: 색상=방향, 밝기=크기


In [ ]:
# 섹션 5 — Farneback Optical Flow 추출기 (이동 벡터 보정 포함)
# 각 tracked 톱니의 연속 ROI 프레임 간 밀집 Flow를 계산합니다.
# 컨베이어 이동 벡터를 차감해 순수 회전 성분만 추출합니다.
# 반환값: (H, W, 2) float32 flow 맵

class FlowExtractor:
    def __init__(self):
        self._prev:Dict[int,np.ndarray]={}
        self.params=dict(pyr_scale=0.5,levels=3,winsize=15,
                          iterations=3,poly_n=5,poly_sigma=1.2,flags=0)

    def compute(self,track_id:int,roi:np.ndarray,
                translation:Tuple)->Optional[np.ndarray]:
        if track_id not in self._prev:
            self._prev[track_id]=roi.copy(); return None
        prev=self._prev[track_id]
        if prev.shape!=roi.shape:
            prev=cv2.resize(prev,roi.shape[::-1])
        flow=cv2.calcOpticalFlowFarneback(prev,roi,None,**self.params)
        dx,dy=translation
        flow[...,0]-=dx; flow[...,1]-=dy   # 컨베이어 이동 성분 제거
        self._prev[track_id]=roi.copy()
        return flow

    def flush(self,track_id:int):
        self._prev.pop(track_id,None)

def flow_to_rgb(flow:np.ndarray)->np.ndarray:
    mag,ang=cv2.cartToPolar(flow[...,0],flow[...,1])
    hsv=np.zeros((*flow.shape[:2],3),dtype=np.uint8)
    hsv[...,0]=ang*180/np.pi/2; hsv[...,1]=255
    hsv[...,2]=cv2.normalize(mag,None,0,255,cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB)


In [ ]:
# 섹션 5 — Optical Flow 계산 및 시각화
# 전체 프레임에 대해 검출 → 추적 → ROI → Flow 파이프라인을 실행합니다.
# 샘플 Flow 맵(색상=방향, 밝기=크기)과 시계열 크기 추이를 출력합니다.
# ※ 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

flow_extractor=FlowExtractor(); roi_module_s5=DynamicROI(); tracker_s5=IoUTracker()
flow_results=[]; mag_per_frame=[]

for fi,frame in enumerate(FRAMES):
    dets=detect_objects(frame); tracked=tracker_s5.update(dets)
    frame_mag=[]
    for t in tracked:
        if not t['confirmed']: continue
        roi_res=roi_module_s5.crop(t['track_id'],t['bbox'],frame)
        flow=flow_extractor.compute(t['track_id'],roi_res['roi'],roi_res['translation'])
        if flow is not None:
            mag=np.mean(np.sqrt(flow[...,0]**2+flow[...,1]**2))
            frame_mag.append(mag)
            flow_results.append((fi,t['track_id'],flow))
    mag_per_frame.append(np.mean(frame_mag) if frame_mag else 0.0)

sample_flows=flow_results[::max(1,len(flow_results)//6)][:6]
if sample_flows:
    n_cols=min(3,len(sample_flows)); n_rows=(len(sample_flows)+n_cols-1)//n_cols
    fig,axes=plt.subplots(n_rows,n_cols,figsize=(5*n_cols,4*n_rows))
    axes=np.array(axes).flatten()
    for ax,(fi,tid,flow) in zip(axes,sample_flows):
        mag=np.mean(np.sqrt(flow[...,0]**2+flow[...,1]**2))
        ax.imshow(flow_to_rgb(flow))
        ax.set_title(f'Frame {fi}  ID={tid}  mag={mag:.2f}',fontsize=9); ax.axis('off')
    for ax in axes[len(sample_flows):]: ax.axis('off')
    plt.suptitle('[Section 5] Optical Flow  |  Hue=Direction  Brightness=Magnitude',fontsize=11,fontweight='bold')
    plt.tight_layout(); plt.show()

if mag_per_frame:
    mag_arr=np.array(mag_per_frame); threshold=np.mean(mag_arr)*0.3
    fig,ax=plt.subplots(figsize=(14,4))
    ax.plot(mag_arr,color='#185FA5',lw=1.5,label='Mean Flow Magnitude')
    ax.axhline(threshold,color='#A32D2D',ls='--',lw=1.5,label=f'Temp threshold={threshold:.2f}')
    ax.fill_between(range(len(mag_arr)),threshold,mag_arr,
                    where=(mag_arr<threshold),color='#A32D2D',alpha=0.2,label='Anomaly suspect')
    ax.set_xlabel('Frame'); ax.set_ylabel('Flow Magnitude')
    ax.set_title('Flow Magnitude Trend  |  Near 0 = stopped  |  Set real threshold after data collection')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f'평균:{mag_arr.mean():.3f}  최대:{mag_arr.max():.3f}  최소:{mag_arr.min():.3f}')


---
# 섹션 6. CNN → GRU 분류

## 아키텍처

```
CNN : Flow 맵 (H×W×2) → 64차원 특징 벡터 (한 프레임)
GRU : 20프레임 시계열 → Class 0(정상) / 1(의심) / 2(이탈)
```

## 현재 단계 안내

**현재 단계**: 학습 전이므로 분류 결과는 의미 없습니다.
파이프라인 구조가 오류 없이 동작하는지만 확인합니다.
현장 데이터 수집 후 학습하면 실제 판정이 가능합니다.


In [ ]:
# 섹션 6 — CNN, GRU, SequenceBuffer 정의 및 텐서 형상 확인
# CNN    : 2채널 Flow 맵 → 64차원 특징 벡터 (프레임 1장)
# GRU    : 20개 특징 벡터 시계열 → 회전 상태 분류 (0/1/2)
# Buffer : 트랙별 독립 슬라이딩 윈도우 (SEQ_LEN 충족 시 배열 반환)

class FlowCNN(nn.Module):
    # Flow 맵(2ch) → 64차원 특징 벡터
    def __init__(self,out_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(2,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,out_dim),nn.ReLU())
    def forward(self,x): return self.net(x)

class RotationGRU(nn.Module):
    # 20프레임 시계열 → Class 0(정상) / 1(의심) / 2(이탈)
    def __init__(self,in_dim=64,hidden=128,n_classes=3):
        super().__init__()
        self.gru=nn.GRU(in_dim,hidden,num_layers=2,batch_first=True,dropout=0.3)
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),
                                 nn.Dropout(0.3),nn.Linear(64,n_classes))
    def forward(self,x):
        _,h=self.gru(x); return self.head(h[-1])

class SequenceBuffer:
    # 트랙별 독립 슬라이딩 윈도우 — CNN 특징 벡터 누적
    SEQ_LEN=20
    def __init__(self): self._bufs:Dict[int,deque]={}
    def push(self,track_id:int,feat:np.ndarray)->Optional[np.ndarray]:
        if track_id not in self._bufs:
            self._bufs[track_id]=deque(maxlen=self.SEQ_LEN)
        self._bufs[track_id].append(feat)
        if len(self._bufs[track_id])==self.SEQ_LEN:
            return np.stack(list(self._bufs[track_id]))
        return None
    def fill_rate(self,track_id:int)->float:
        if track_id not in self._bufs: return 0.0
        return len(self._bufs[track_id])/self.SEQ_LEN
    def flush(self,track_id:int): self._bufs.pop(track_id,None)

def flow_to_tensor(flow:np.ndarray)->torch.Tensor:
    resized=cv2.resize(flow,(64,64))
    t=torch.from_numpy(resized.transpose(2,0,1)).float().unsqueeze(0)
    return torch.tanh(t/5.0)

cnn_model=FlowCNN(out_dim=64).to(DEVICE)
gru_model=RotationGRU(in_dim=64,hidden=128).to(DEVICE)

print('모델 파라미터')
print(f'  CNN: {sum(p.numel() for p in cnn_model.parameters()):,}')
print(f'  GRU: {sum(p.numel() for p in gru_model.parameters()):,}')
dummy=torch.randn(1,2,64,64).to(DEVICE)
with torch.no_grad():
    fo=cnn_model(dummy)
    co=gru_model(fo.unsqueeze(0).repeat(1,20,1))
print(f'  CNN: {dummy.shape} → {fo.shape}')
print(f'  GRU: {fo.unsqueeze(0).repeat(1,20,1).shape} → {co.shape}')


In [ ]:
# 섹션 6 — 전체 CNN+GRU 파이프라인 실행
# 모든 프레임에 대해: ROI → Flow → CNN → SequenceBuffer → GRU 순으로 처리합니다.
# GRU 버퍼 충전율 및 클래스별 확률 추이를 출력합니다.
# ※ 학습 전 모델이므로 분류 결과는 구조 확인용입니다.
# ※ 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

flow_ext_s6=FlowExtractor(); roi_mod_s6=DynamicROI()
tracker_s6=IoUTracker(); seq_buf_s6=SequenceBuffer()
cnn_model.eval(); gru_model.eval()
CLASS_NAMES=['Class 0: Normal','Class 1: Suspect','Class 2: Derailed']
CLASS_COLORS=['#0F6E56','#EF9F27','#A32D2D']
frame_log=[]

for fi,frame in enumerate(FRAMES):
    dets=detect_objects(frame); tracked=tracker_s6.update(dets)
    for t in tracked:
        if not t['confirmed']: continue
        tid=t['track_id']
        roi_res=roi_mod_s6.crop(tid,t['bbox'],frame)
        flow=flow_ext_s6.compute(tid,roi_res['roi'],roi_res['translation'])
        if flow is None: continue
        ft=flow_to_tensor(flow).to(DEVICE)
        with torch.no_grad():
            feat=cnn_model(ft).squeeze(0).cpu().numpy()
        seq=seq_buf_s6.push(tid,feat); fill=seq_buf_s6.fill_rate(tid)
        if seq is not None:
            seq_t=torch.tensor(seq,dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                probs=F.softmax(gru_model(seq_t),dim=1).squeeze(0).cpu().numpy()
            frame_log.append((fi,fill,int(np.argmax(probs)),probs))
        else:
            frame_log.append((fi,fill,None,None))

fig,axes=plt.subplots(2,1,figsize=(14,8))
fis=[r[0] for r in frame_log]; fills=[r[1] for r in frame_log]
axes[0].plot(fis,fills,color='#185FA5',lw=2)
axes[0].fill_between(fis,fills,alpha=0.15,color='#185FA5')
axes[0].axhline(1.0,color='gray',ls='--',alpha=0.4)
axes[0].set_ylabel('Buffer Fill Rate'); axes[0].set_ylim(-0.05,1.15)
axes[0].set_title(f'GRU Buffer Fill Rate  |  Classification starts after SEQ_LEN={SequenceBuffer.SEQ_LEN} frames')
axes[0].grid(alpha=0.3)

judged=[(r[0],r[2],r[3]) for r in frame_log if r[2] is not None]
if judged:
    jf=[x[0] for x in judged]
    for ci,(name,color) in enumerate(zip(CLASS_NAMES,CLASS_COLORS)):
        axes[1].plot(jf,[x[2][ci] for x in judged],label=name,color=color,lw=1.5)
    axes[1].axhline(0.5,color='gray',ls='--',alpha=0.4)
    axes[1].set_ylim(-0.05,1.05); axes[1].set_ylabel('Class Probability')
    axes[1].set_xlabel('Frame'); axes[1].set_title('Per-class Probability (untrained — structural check only)')
    axes[1].legend()

plt.suptitle('[Section 6] CNN+GRU Pipeline Verification',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()
print(f'처리: {len(frame_log)}개  판정: {len(judged)}개')
print()
print('다음 단계: 현장 정상/이상 영상 수집 → CNN+GRU 학습')


---
# 섹션 7-A. 카메라 1 — 후크 적재 판정 모델

## 역할

카메라 2가 YOLO + CNN + GRU로 톱니 회전 상태를 판정하듯,
카메라 1도 YOLO로 후크에 도장 대상이 걸려 있는지를 판정합니다.

```
카메라 1 프레임
    ↓ YOLO 검출 (cam1 모델)
후크 영역에 객체가 있는가?
    ├── 검출됨  → LOADED  → 후크 상태 큐에 push
    └── 없음    → EMPTY   → 후크 상태 큐에 push
```

## 모드 선택

| CAM1_MODE | 상황 |
|-----------|------|
| `'builtin'` | 내장 합성 이미지로 구조 테스트 (현장 영상 없을 때) |
| `'same_frames'` | 카메라 2와 동일한 FRAMES 사용 (단일 카메라 테스트) |
| `'schedule'` | 수동 스케줄 입력 (기존 방식, 빠른 테스트용) |

## 검출 방식

카메라 2의 `detect_objects()`와 동일한 파인튜닝 모델을 사용합니다.
후크 영역에 gear/hook bbox가 1개 이상 검출되면 LOADED, 없으면 EMPTY로 판정합니다.

> 실제 현장에서는 카메라 1 전용 이미지로 별도 파인튜닝을 권장합니다.
> 지금은 카메라 2와 동일한 모델로 구조를 검증합니다.


In [ ]:
# 섹션 7-A — 카메라 1 후크 적재 판정 모델
#
# CAM1_MODE에 따라 카메라 1 판정 방식을 선택합니다.
# 'builtin' 또는 'same_frames': YOLO 검출 결과로 LOADED/EMPTY 자동 판정
# 'schedule': 수동 스케줄 입력 (기존 방식)

import glob as _gl_c1

# ── 여기만 수정하세요 ──────────────────────────
CAM1_MODE = 'same_frames'
# 'builtin'      : 내장 합성 이미지(FRAMES)로 테스트
# 'same_frames'  : 카메라 2와 동일한 FRAMES 사용
# 'schedule'     : 아래 수동 스케줄 사용

CAM1_CONF_THR = 0.35   # 카메라 1 검출 신뢰도 임계값
#   낮출수록 더 많이 LOADED 판정 (민감)
#   높일수록 확실한 경우만 LOADED 판정 (보수적)

# 수동 스케줄 (CAM1_MODE='schedule' 일 때만 사용)
CAM1_MANUAL_SCHEDULE = [
    (0,  'LOADED'),
    (20, 'LOADED'),
    (35, 'EMPTY'),
]
# ───────────────────────────────────────────────


def cam1_detect(frame: 'np.ndarray') -> str:
    # 카메라 1 판정 함수
    # 반환: 'LOADED' 또는 'EMPTY'
    #
    # 카메라 2와 동일한 파인튜닝 모델(detect_objects)을 사용합니다.
    # 후크 영역에 bbox가 1개 이상 검출되면 LOADED, 없으면 EMPTY
    dets = detect_objects(frame)
    return 'LOADED' if len(dets) > 0 else 'EMPTY'


# ── 카메라 1 전체 프레임 판정 실행 ──
# 결과를 CAM1_RESULTS 딕셔너리에 저장합니다.
# {프레임번호: 'LOADED' or 'EMPTY'}
# 섹션 8에서 CAM1_HOOK_SCHEDULE 대신 이 딕셔너리를 사용합니다.

CAM1_RESULTS = {}  # {frame_idx: 'LOADED'/'EMPTY'}

if CAM1_MODE == 'schedule':
    # 수동 스케줄 — 기존 방식
    CAM1_RESULTS = dict(CAM1_MANUAL_SCHEDULE)
    print(f'카메라 1: 수동 스케줄 사용 ({len(CAM1_RESULTS)}개 이벤트)')
    for fi, st in sorted(CAM1_RESULTS.items()):
        print(f'  프레임 {fi}: {st}')

else:
    # 'builtin' / 'same_frames' — YOLO 자동 판정
    loaded_cnt = 0
    empty_cnt  = 0

    print(f'카메라 1: YOLO 자동 판정 (mode={CAM1_MODE}, conf_thr={CAM1_CONF_THR})')
    print(f'  전체 {len(FRAMES)}프레임 처리 중...')

    for fi, frame in enumerate(FRAMES):
        status = cam1_detect(frame)
        CAM1_RESULTS[fi] = status
        if status == 'LOADED':
            loaded_cnt += 1
        else:
            empty_cnt += 1

    print(f'  처리 완료: {len(CAM1_RESULTS)}프레임')
    print(f'  LOADED: {loaded_cnt}프레임 ({loaded_cnt/len(FRAMES)*100:.1f}%)')
    print(f'  EMPTY : {empty_cnt}프레임 ({empty_cnt/len(FRAMES)*100:.1f}%)')
    print()

    # 판정 결과 시각화 — 프레임별 LOADED/EMPTY 분포
    fig, ax = plt.subplots(figsize=(14, 2.5))
    fis_loaded = [fi for fi, st in CAM1_RESULTS.items() if st == 'LOADED']
    fis_empty  = [fi for fi, st in CAM1_RESULTS.items() if st == 'EMPTY']
    ax.bar(fis_loaded, [1]*len(fis_loaded), color='#185FA5',
           alpha=0.8, label=f'LOADED ({loaded_cnt})')
    ax.bar(fis_empty,  [1]*len(fis_empty),  color='#9c9a92',
           alpha=0.5, label=f'EMPTY ({empty_cnt})')
    ax.set_xlabel('Frame')
    ax.set_yticks([])
    ax.set_title(
        f'[Section 7-A] Camera 1 Detection  |  '
        f'Blue=LOADED  Gray=EMPTY  conf_thr={CAM1_CONF_THR}'
    )
    ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()

    # 샘플 프레임 시각화 — LOADED/EMPTY 각 3장씩
    sample_loaded = [fi for fi, st in CAM1_RESULTS.items() if st == 'LOADED'][:3]
    sample_empty  = [fi for fi, st in CAM1_RESULTS.items() if st == 'EMPTY'][:3]
    samples = [(fi, 'LOADED', '#185FA5') for fi in sample_loaded] + \
              [(fi, 'EMPTY',  '#9c9a92') for fi in sample_empty]

    if samples:
        fig, axes = plt.subplots(1, len(samples),
                                  figsize=(len(samples)*3.5, 3.5))
        if len(samples) == 1: axes = [axes]
        for ax, (fi, status, color) in zip(axes, samples):
            frame = FRAMES[fi]
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            dets  = detect_objects(frame)
            ax.imshow(rgb)
            for d in dets:
                x1,y1,x2,y2 = d['bbox']
                ax.add_patch(patches.Rectangle(
                    (x1,y1), x2-x1, y2-y1,
                    lw=2, edgecolor='#EF9F27', facecolor='none'))
            ax.set_title(
                f'Frame {fi}\n{status}  ({len(dets)} det)',
                fontsize=9, color=color, fontweight='bold')
            ax.axis('off')
        plt.suptitle(
            '[Section 7-A] Camera 1 sample frames  |  '
            'Orange box = detected gear/hook',
            fontsize=10, fontweight='bold')
        plt.tight_layout(); plt.show()

    if loaded_cnt == 0:
        print('LOADED 판정 0개 — 모든 프레임이 EMPTY로 판정되었습니다.')
        print('  원인 1: 파인튜닝 모델이 없거나 conf_thr 가 너무 높음')
        print(f'  조치:   CAM1_CONF_THR 을 낮추세요 (현재: {CAM1_CONF_THR})')
        print('  조치:   또는 CAM1_MODE = "schedule" 로 변경해 테스트하세요')
    else:
        print('카메라 1 판정 완료. 섹션 7 시나리오 테스트 또는 섹션 8로 진행하세요.')


---
# 섹션 7. 후크 상태 큐

## 역할

카메라 1과 카메라 2는 컨베이어 라인 위에 **물리적으로 떨어진 위치**에 설치됩니다.

```
컨베이어 이동 방향 →
 ──────────────────────────────────────────────────────────
    [빈]  [적재]            [빈]  [적재]
      ↓     ↓                        ↓     ↓
  📷 카메라 1         거리 d         📷 카메라 2
  후크 적재 여부 판정          톱니 회전 상태 판정
  → LOADED / EMPTY    ←──────────────→   → 큐 조회 후 최종 판정
                      이동 시간 = d / v
```

**문제**: 카메라 1이 `LOADED`를 감지한 그 후크가 카메라 2 앞에 도착하는 시점은
컨베이어 속도와 두 카메라 간 거리에 따라 달라집니다.

**해결**: 카메라 1 감지 시각 + `d / v` = **카메라 2 ETA** 를 계산해 큐에 저장.
카메라 2가 판정할 때 ETA ± 허용 오차 내에 있는 항목만 꺼내 매칭합니다.

## 설정값 의미

| 파라미터 | 의미 | 조정 기준 |
|----------|------|----------|
| `CAM_DISTANCE_M` | 두 카메라 간 물리적 거리 (m) | 현장 실측값으로 교체 |
| `CONVEYOR_SPEED_MPS` | 컨베이어 이동 속도 (m/s) | 인코더 또는 속도계 측정값 |
| `QUEUE_TOLERANCE_S` | ETA 허용 오차 (초) | 속도 변동이 클수록 크게 |

이동 시간 = `CAM_DISTANCE_M / CONVEYOR_SPEED_MPS`

예: 거리 0.5m, 속도 0.1m/s → 이동 시간 **5초**, 허용 오차 ±0.8초 내에 도착한 후크만 매칭

## 판정 매트릭스

| 카메라 1 | 카메라 2 | 결과 |
|----------|----------|------|
| LOADED | 정상 | PASS |
| LOADED | 의심 | WARN |
| LOADED | 이탈 | ALARM → 라인 정지 |
| EMPTY | 어느 쪽 | SKIP |

**안전 측 원칙**: 큐 매칭 실패(타임아웃) → 기본값 `LOADED` 처리 (미검출이 오경보보다 안전)


In [ ]:
# 섹션 7 — 후크 상태 큐 구현
#
# 카메라 1이 후크 적재 상태를 감지하면 큐에 push합니다.
# 이동 시간(= CAM_DISTANCE_M / CONVEYOR_SPEED_MPS) 후
# 카메라 2가 판정 시점에 큐를 pop해 해당 후크의 적재 여부를 조회합니다.
#
# ETA(예상 도착 시간) = detected_at + travel_time
# 허용 오차(QUEUE_TOLERANCE_S) 범위 내에 들어온 항목만 유효하게 매칭합니다.

# ── 여기만 수정하세요 ──────────────────────────
# 현장 실측값으로 교체하세요.

CAM_DISTANCE_M     = 0.5   # 두 카메라 간 물리적 거리 (m) — 현장 줄자 실측
CONVEYOR_SPEED_MPS = 0.1   # 컨베이어 이동 속도 (m/s)  — 인코더 또는 속도계
QUEUE_TOLERANCE_S  = 0.8   # ETA 허용 오차 (초)         — 속도 변동 여유

# 이동 시간 미리보기 (실행 시 자동 계산)
_travel_preview = CAM_DISTANCE_M / CONVEYOR_SPEED_MPS
print(f'카메라 간 이동 시간: {_travel_preview:.1f}초  '
      f'(= {CAM_DISTANCE_M}m / {CONVEYOR_SPEED_MPS}m/s)')
print(f'매칭 허용 범위: ETA ± {QUEUE_TOLERANCE_S}초')
print()

# ───────────────────────────────────────────────
@dataclass
class HookEvent:
    hook_id:str; status:str; detected_at:float; eta_cam2:float; ttl:float

class HookQueue:
    def __init__(self,distance_m,speed_mps,tolerance_s):
        self.travel_time=distance_m/speed_mps
        self.tolerance=tolerance_s
        self._queue:List[HookEvent]=[]; self._cnt=0

    def push(self,status:str,detected_at:float):
        self._cnt+=1
        self._queue.append(HookEvent(
            f'HK-{self._cnt:04d}',status,detected_at,
            detected_at+self.travel_time,self.travel_time*1.5))

    def pop(self,query_time:float)->Tuple[str,Optional[str]]:
        self._queue=[e for e in self._queue
                     if query_time-e.detected_at<=e.ttl]
        if not self._queue: return 'LOADED',None
        best=min(self._queue,key=lambda e:abs(e.eta_cam2-query_time))
        if abs(best.eta_cam2-query_time)>self.tolerance: return 'LOADED',None
        self._queue.remove(best); return best.status,best.hook_id

def final_decision(hook_status:str,rotation_class:int)->Dict:
    if hook_status=='EMPTY':
        return {'decision':'SKIP','action':'빈 후크 — 무시','urgent':False}
    if rotation_class==0:
        return {'decision':'PASS','action':'정상 — 계속 운행','urgent':False}
    elif rotation_class==1:
        return {'decision':'WARN','action':'의심 — HMI 경고','urgent':False}
    else:
        return {'decision':'ALARM','action':'이탈 — PLC 즉시 정지','urgent':True}


In [ ]:
# 섹션 7 — 후크 상태 큐 4가지 시나리오 테스트
# 카메라 1 후크 상태와 카메라 2 회전 클래스의 네 조합을 시뮬레이션합니다.
# 각 케이스에서 최종 판정 로직이 예상값을 정확히 출력하는지 검증합니다.

scenarios=[
    (0.0,'LOADED',5.0,0,'PASS', '도장 대상 있음 + 정상 회전'),
    (1.0,'LOADED',6.0,1,'WARN', '도장 대상 있음 + 의심'),
    (2.0,'LOADED',7.0,2,'ALARM','도장 대상 있음 + 이탈 → 경보'),
    (3.0,'EMPTY', 8.0,2,'SKIP', '빈 후크 + 이탈처럼 보임 → 무시'),
]
q=HookQueue(CAM_DISTANCE_M,CONVEYOR_SPEED_MPS,QUEUE_TOLERANCE_S)
for cam1_t,status,_,_,_,_ in scenarios: q.push(status,cam1_t)

print('='*60); print('후크 상태 큐 — 4가지 시나리오'); print('='*60)
results=[]
for cam1_t,status,cam2_t,rot_cls,expected,desc in scenarios:
    hook_status,_=q.pop(cam2_t)
    result=final_decision(hook_status,rot_cls)
    ok='✅' if result['decision']==expected else '❌'
    results.append(ok)
    print(f'{ok} {desc}')
    print(f'   카메라1:{status:6s}  GRU:Class{rot_cls}  '
          f'조회:{hook_status:6s}  판정:{result["decision"]:5s}  (예상:{expected})')
    if result['urgent']: print(f'   ⚡ {result["action"]}')
    print()
print(f'테스트: {results.count("✅")}/{len(results)} 통과')


---
# 섹션 8. 전체 파이프라인 통합 실행

## 개요

지금까지 독립 검증한 모듈들을 하나의 루프로 연결합니다.
매 프레임은 다음 순서로 처리됩니다:
YOLO 검출 → Tracker → ROI 추출 → Optical Flow → CNN → GRU → 후크 상태 큐 → 최종 판정


In [ ]:
# 섹션 8 — 전체 프레임에 대한 통합 파이프라인 실행
#
# 카메라 1 판정 결과(CAM1_RESULTS)를 자동으로 사용합니다.
# 섹션 7-A에서 YOLO 자동 판정 또는 수동 스케줄 중 하나를 선택해 실행하면
# CAM1_RESULTS 딕셔너리가 채워지고 여기서 그대로 사용됩니다.
# 섹션 2의 YOLO_MODE 설정은 변경되지 않습니다.

# ── 카메라 1 판정 결과 확인 ──
if 'CAM1_RESULTS' not in dir() or len(CAM1_RESULTS) == 0:
    print('CAM1_RESULTS 가 없습니다. 섹션 7-A를 먼저 실행하세요.')
    raise RuntimeError('섹션 7-A 먼저 실행 필요')

print(f'카메라 1 판정 결과 확인: {len(CAM1_RESULTS)}프레임')
loaded_n = sum(1 for v in CAM1_RESULTS.values() if v == 'LOADED')
empty_n  = sum(1 for v in CAM1_RESULTS.values() if v == 'EMPTY')
print(f'  LOADED: {loaded_n}  EMPTY: {empty_n}')
print()

# ── 여기만 수정하세요 ──────────────────────────
FPS = 10   # 프레임 속도 가정 (초당 프레임 수)
           # 실제 카메라 fps에 맞게 조정하세요
# ───────────────────────────────────────────────

tracker_final    = IoUTracker()
roi_final        = DynamicROI()
flow_final       = FlowExtractor()
seq_buf_final    = SequenceBuffer()
hook_queue_final = HookQueue(CAM_DISTANCE_M, CONVEYOR_SPEED_MPS, QUEUE_TOLERANCE_S)

cnn_model.eval(); gru_model.eval()
run_log = []

for fi, frame in enumerate(FRAMES):
    t_now = float(fi) / FPS  # 프레임 번호 → 실제 시각(초)

    # 카메라 1 판정 결과를 큐에 push
    # CAM1_RESULTS: {프레임번호: 'LOADED'/'EMPTY'}
    # 섹션 7-A에서 YOLO 자동 판정 또는 수동 스케줄로 채워진 값 사용
    if fi in CAM1_RESULTS:
        hook_queue_final.push(CAM1_RESULTS[fi], detected_at=t_now)

    dets    = detect_objects(frame)
    tracked = tracker_final.update(dets)
    frame_result = {
        'fi': fi, 't': t_now,
        'cam1_status': CAM1_RESULTS.get(fi, None),
        'n_detected': len([t for t in tracked if t['confirmed']]),
        'buffer_fill': 0.0, 'rot_class': None, 'rot_probs': None,
        'hook_status': None, 'decision': None
    }

    for t in tracked:
        if not t['confirmed']: continue
        tid = t['track_id']
        roi_res = roi_final.crop(tid, t['bbox'], frame)
        flow    = flow_final.compute(tid, roi_res['roi'], roi_res['translation'])
        if flow is None: continue
        ft = flow_to_tensor(flow).to(DEVICE)
        with torch.no_grad():
            feat = cnn_model(ft).squeeze(0).cpu().numpy()
        seq  = seq_buf_final.push(tid, feat)
        frame_result['buffer_fill'] = seq_buf_final.fill_rate(tid)
        if seq is not None:
            seq_t = torch.tensor(
                seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                probs = F.softmax(
                    gru_model(seq_t), dim=1).squeeze(0).cpu().numpy()
            rot_cls = int(np.argmax(probs))
            frame_result['rot_class']  = rot_cls
            frame_result['rot_probs']  = probs
            hook_status, _ = hook_queue_final.pop(t_now + 5.0)
            frame_result['hook_status'] = hook_status
            frame_result['decision']    = final_decision(hook_status, rot_cls)
    run_log.append(frame_result)

judged_count = sum(1 for r in run_log if r['decision'] is not None)
print(f'처리 완료: {len(run_log)}프레임  판정: {judged_count}프레임')
print(f'카메라 1 소스: {CAM1_MODE if "CAM1_MODE" in dir() else "schedule"}')


In [ ]:
# 섹션 8 — 전체 파이프라인 결과 시각화
# 4개 패널 차트: (1) GRU 버퍼 충전율  (2) 카메라 1 후크 이벤트
#               (3) 최종 판정 분포    (4) ALARM/WARN 발생 프레임
# 이후 실행 요약과 다음 단계 체크리스트를 출력합니다.
# ※ 차트 제목/레이블은 영문으로 표기합니다 (한글 깨짐 방지).

DC={'PASS':'#0F6E56','WARN':'#EF9F27','ALARM':'#A32D2D','SKIP':'#9c9a92',None:'#D3D1C7'}
DL={'PASS':0,'WARN':1,'ALARM':2,'SKIP':-1,None:-2}
fis=[r['fi'] for r in run_log]; fills=[r['buffer_fill'] for r in run_log]

fig,axes=plt.subplots(4,1,figsize=(15,14),sharex=True)

axes[0].plot(fis,fills,color='#185FA5',lw=2)
axes[0].fill_between(fis,fills,alpha=0.15,color='#185FA5')
axes[0].axhline(1.0,color='gray',ls='--',alpha=0.4)
axes[0].set_ylabel('Buffer Fill Rate'); axes[0].set_ylim(-0.05,1.15)
axes[0].set_title(f'(1) GRU Buffer Fill Rate  |  Classification starts after SEQ_LEN={SequenceBuffer.SEQ_LEN} frames')
axes[0].grid(alpha=0.2)

cam1_fi=[fi for fi,_ in CAM1_HOOK_SCHEDULE]
cam1_st=[st for _,st in CAM1_HOOK_SCHEDULE]
axes[1].vlines(cam1_fi,0,1,colors=['#185FA5' if s=='LOADED' else '#9c9a92' for s in cam1_st],lw=4)
for fi,st in CAM1_HOOK_SCHEDULE:
    axes[1].text(fi,1.05,st,ha='center',fontsize=9,
                  color='#185FA5' if st=='LOADED' else '#9c9a92')
axes[1].set_ylabel('Camera 1'); axes[1].set_ylim(-0.1,1.4); axes[1].set_yticks([])
axes[1].set_title('(2) Camera 1 Hook Events  |  Blue=LOADED  Gray=EMPTY'); axes[1].grid(alpha=0.2)

judged_rows=[(r['fi'],r['decision']['decision']) for r in run_log if r['decision']]
if judged_rows:
    jfi=[x[0] for x in judged_rows]; jdec=[x[1] for x in judged_rows]
    axes[2].scatter(jfi,[DL[d] for d in jdec],c=[DC[d] for d in jdec],s=60,zorder=4)
    axes[2].set_yticks([-1,0,1,2]); axes[2].set_yticklabels(['SKIP','PASS','WARN','ALARM'])
    for level,color in [(-1,'#F1EFE8'),(0,'#E1F5EE'),(1,'#FAEEDA'),(2,'#FCEBEB')]:
        axes[2].axhspan(level-0.4,level+0.4,alpha=0.15,color=color)
axes[2].set_ylabel('Decision'); axes[2].set_title('(3) Final Decision  (untrained — structural check only)'); axes[2].grid(alpha=0.2)

alarm_fi=[r['fi'] for r in run_log if r['decision'] and r['decision']['decision']=='ALARM']
warn_fi =[r['fi'] for r in run_log if r['decision'] and r['decision']['decision']=='WARN']
axes[3].bar(fis,[1]*len(fis),color='#E1F5EE',alpha=0.5)
if warn_fi:  axes[3].bar(warn_fi, [1]*len(warn_fi), color='#EF9F27',alpha=0.8,label='WARN')
if alarm_fi: axes[3].bar(alarm_fi,[1]*len(alarm_fi),color='#A32D2D',alpha=0.9,label='ALARM')
axes[3].set_yticks([]); axes[3].set_xlabel('Frame')
axes[3].set_title('(4) ALARM / WARN Frame Distribution'); axes[3].legend(loc='upper right')

plt.suptitle('[Section 8] Full Pipeline Integration',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

print(); print('='*60); print('실행 요약'); print('='*60)
print(f'처리: {len(run_log)}프레임  판정: {judged_count}프레임')
dc={}
for r in run_log:
    if r['decision']:
        d=r['decision']['decision']; dc[d]=dc.get(d,0)+1
for d,cnt in sorted(dc.items()):
    icon={'PASS':'✅','WARN':'⚠️','ALARM':'🚨','SKIP':'⏭'}.get(d,' ')
    print(f'  {icon} {d:5s}: {cnt}프레임')

print(); print('='*60); print('다음 단계 체크리스트'); print('='*60)
print('□ 섹션 1: 실제 영상으로 MODE 변경')
print('□ 섹션 2-B~2-E: gear/hook 파인튜닝')
print('□ 섹션 2: YOLO_MODE=custom, YOLO_WEIGHTS=best.pt')
print('□ 섹션 3: IOU_THRESH/MAX_LOST 현장 속도에 맞게')
print('□ 섹션 4: PADDING 톱니 크기에 맞게')
print('□ 섹션 6: 정상/이상 영상 수집 후 CNN+GRU 학습')
print('□ 섹션 7: CAM_DISTANCE_M, CONVEYOR_SPEED_MPS 실측값')
print('□ 섹션 8: CAM1_HOOK_SCHEDULE → 실제 카메라 1 스트림')


---
---

# ══════════════════════════════════════════
# 스마트 도장라인 AI v5 — 카메라 1 파이프라인
# ══════════════════════════════════════════

## 카메라 1의 역할

카메라 2(톱니 회전 판정)와 달리 카메라 1은 **후크에 도장 대상이 걸려 있는지**를 판단합니다.

```
카메라 1이 보는 장면:

  체인 ━━━━━━━━━━━━━━━━━━━━━
        │  후크 암
        ◎  톱니 (gear)
        │
        ┊  철사
        │
     [도장 객체]  ← LOADED: 있음 / EMPTY: 없음
  ━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## v5 파이프라인 구성

```
[섹션 C1-0] 카메라 1 환경 설정
[섹션 C1-1] 카메라 1 이미지 로더 (내장 / 폴더 / 영상)
[섹션 C1-2] 카메라 1 YOLO 검출 (클래스: workpiece)
   ├─ [섹션 C1-2B] Grounding DINO 자동 라벨링
   ├─ [섹션 C1-2C] 라벨 검수
   ├─ [섹션 C1-2D] YOLOv8 파인튜닝
   └─ [섹션 C1-2E] 성능 검증
[섹션 C1-3] LOADED/EMPTY 판정 + 후크 큐 연동
```

## 카메라 2와의 차이

| 항목 | 카메라 2 | 카메라 1 |
|------|----------|----------|
| 검출 대상 | gear, hook (톱니) | workpiece (도장 객체) |
| 판정 결과 | Class 0/1/2 (회전 상태) | LOADED / EMPTY |
| Tracker 필요 | 필요 | 불필요 |
| Flow/CNN/GRU | 필요 | 불필요 |
| 데이터 폴더 | `dataset/` | `dataset_cam1/` |
| 모델 변수 | `FINETUNED_PT` | `CAM1_FINETUNED_PT` |


---
# 섹션 C1-0. 카메라 1 환경 설정

섹션 0의 패키지/라이브러리가 이미 로드되어 있다면 이 섹션은 건너뛰어도 됩니다.
런타임 재시작 후에는 섹션 0을 먼저 실행한 뒤 이 섹션으로 돌아오세요.


In [ ]:
# 섹션 C1-0 — 카메라 1 전용 폴더 생성
# 카메라 2 데이터(dataset/)와 분리해서 dataset_cam1/에 저장합니다.

import os
os.makedirs('dataset_cam1/images/train', exist_ok=True)
os.makedirs('dataset_cam1/labels/train', exist_ok=True)
os.makedirs('dataset_cam1/images/val',   exist_ok=True)
os.makedirs('dataset_cam1/labels/val',   exist_ok=True)
os.makedirs('/content/cam1_frames',      exist_ok=True)

print('카메라 1 폴더 생성 완료')
print('  dataset_cam1/images/train')
print('  dataset_cam1/labels/train')
print('  /content/cam1_frames')


---
# 섹션 C1-1. 카메라 1 이미지 로더

## 내장 이미지 구조

카메라 1은 후크 아래 매달린 도장 객체 유무를 판단합니다.

```
LOADED 프레임:              EMPTY 프레임:
  체인 ━━━━━━━━━━━━━          체인 ━━━━━━━━━━━━━
      │ 후크 암                   │ 후크 암
      ◎ 톱니                      ◎ 톱니
      │ 철사                      │ (철사만)
   ┌──────┐
   │ 도장 │  ← workpiece
   └──────┘
  바닥 ━━━━━━━━━━━━━━          바닥 ━━━━━━━━━━━━━
```

## 설정

| 파라미터 | 설명 |
|----------|------|
| `CAM1_MODE` | `'builtin'` / `'folder'` / `'video'` |
| `CAM1_MAX_FRAMES` | 생성할 총 프레임 수 (기본 300) |
| `CAM1_LOADED_RATIO` | LOADED 비율 (기본 0.7 = 70%) |


In [ ]:
# 섹션 C1-1 — 카메라 1 이미지 로더 설정 및 내장 이미지 생성기

# ── 여기만 수정하세요 ──────────────────────────
CAM1_MODE          = 'builtin'
CAM1_FOLDER_PATH   = '/content/cam1_frames_real'
CAM1_VIDEO_PATH    = '/content/cam1_sample.mp4'
CAM1_MAX_FRAMES    = 300          # 총 생성 프레임 수
CAM1_LOADED_RATIO  = 0.7          # LOADED 비율 (0.7 = 70%)
CAM1_FRAME_W       = 640
CAM1_FRAME_H       = 480
CAM1_SAVE_DIR      = '/content/cam1_frames'
# ───────────────────────────────────────────────


def make_cam1_builtin_frames(n=300, w=640, h=480, loaded_ratio=0.7):
    """
    카메라 1 합성 프레임 생성기 (기본 300장)

    핵심 설계: LOADED/EMPTY를 프레임마다 결정하지 않고 후크 단위로 결정합니다.
      - 하나의 후크가 화면에 있는 동안은 상태가 유지됩니다.
      - 새 후크가 화면에 진입할 때 LOADED/EMPTY를 새로 결정합니다.
      - 실제 현장과 동일하게 도장 대상이 갑자기 사라졌다 나타나지 않습니다.

    장면 구조:
      체인 (상단) → 후크 암 → 톱니(gear) → 철사 → 도장 객체 or 없음

    다양성 요소:
      - 후크 단위 LOADED/EMPTY (loaded_ratio 비율)
      - 후크마다 도장 객체 크기/밝기 랜덤
      - 조명 변화 (배경 밝기 ±15 노이즈)
      - 분진 효과 (랜덤 백색 점)
    """
    import math, random
    import numpy as np
    import cv2

    frames = []
    labels = []   # 프레임별 'LOADED' or 'EMPTY'

    chain_y    = int(h * 0.10)
    gear_y     = int(h * 0.32)
    wire_bot_y = int(h * 0.48)
    floor_y    = int(h * 0.80)
    speed_px   = 5.0            # 컨베이어 이동 속도 (px/프레임)

    # ── 후크 풀 생성 ──────────────────────────────────────────
    # 후크가 화면 오른쪽 밖에서 시작해 왼쪽으로 이동합니다.
    # 각 후크는 생성 시 LOADED/EMPTY 상태가 결정되고 화면에 있는 동안 유지됩니다.
    #
    # 후크 간격: 220px (화면 폭 640px 기준 약 2~3개 동시 표시)
    # n 프레임 동안 이동 거리: n * speed_px = 300 * 5 = 1500px
    # 필요한 후크 수: (1500 + w) / 220 ≈ 10개 (여유분 포함 15개 생성)

    HOOK_INTERVAL = 220   # 후크 간격 (px)
    n_hooks_total = int((n * speed_px + w) / HOOK_INTERVAL) + 4

    hook_pool = []
    for hi in range(n_hooks_total):
        # 시작 x: 화면 오른쪽 밖에서 간격마다 배치
        start_x    = w + hi * HOOK_INTERVAL
        is_loaded  = (random.random() < loaded_ratio)
        gear_r     = random.randint(18, 26)
        obj_w      = random.randint(55, 95)
        obj_h      = random.randint(38, 65)
        obj_bright = random.randint(130, 210)
        hook_pool.append({
            'start_x':   start_x,
            'is_loaded': is_loaded,   # 이 후크의 상태 — 한 번 결정 후 고정
            'gear_r':    gear_r,
            'obj_w':     obj_w,
            'obj_h':     obj_h,
            'obj_bright':obj_bright,
        })

    # ── 프레임 생성 ──────────────────────────────────────────
    for i in range(n):
        base_bright = 50 + int(random.gauss(0, 10))
        base_bright = max(30, min(70, base_bright))
        frame = np.full((h, w, 3), base_bright, dtype=np.uint8)

        # 체인
        cv2.rectangle(frame, (0, chain_y-4), (w, chain_y+10), (90,90,88), -1)
        cv2.rectangle(frame, (0, chain_y-4), (w, chain_y),    (110,110,108), -1)
        for rx in range(20, w, 32):
            cv2.ellipse(frame, (rx, chain_y+3), (12,5), 0, 0, 360, (70,70,68), -1)
            cv2.ellipse(frame, (rx, chain_y+3), (12,5), 0, 0, 360, (50,50,48), 1)

        # 바닥
        cv2.rectangle(frame, (0, floor_y), (w, h), (40,40,38), -1)
        cv2.rectangle(frame, (0, floor_y), (w, floor_y+4), (70,70,68), -1)

        # 이 프레임에서 화면에 보이는 후크들의 상태 수집
        frame_has_loaded = False

        for hc in hook_pool:
            # 프레임 i에서 이 후크의 x 위치
            # 오른쪽에서 왼쪽으로 이동: start_x - i * speed_px
            cx_now = int(hc['start_x'] - i * speed_px)

            # 화면 밖이면 스킵
            if cx_now < -80 or cx_now > w + 80:
                continue

            is_loaded = hc['is_loaded']
            gear_r    = hc['gear_r']

            # 후크 암
            cv2.rectangle(frame,
                (cx_now-8, chain_y+10),
                (cx_now+8, gear_y),
                (80,78,75), -1)

            # 톱니
            cv2.ellipse(frame, (cx_now, gear_y),
                (gear_r, int(gear_r*0.55)), 0, 0, 360, (95,92,88), -1)
            cv2.ellipse(frame, (cx_now, gear_y),
                (gear_r, int(gear_r*0.55)), 0, 0, 360, (60,58,55), 1)
            cv2.ellipse(frame, (cx_now, gear_y),
                (7, 4), 0, 0, 360, (55,53,50), -1)

            # 철사
            cv2.line(frame,
                (cx_now, gear_y + int(gear_r*0.55)),
                (cx_now, wire_bot_y),
                (100,98,95), 1)

            if is_loaded:
                frame_has_loaded = True
                obj_w = hc['obj_w']; obj_h = hc['obj_h']
                ob    = hc['obj_bright']
                ox1   = cx_now - obj_w//2
                oy1   = wire_bot_y
                ox2   = cx_now + obj_w//2
                oy2   = wire_bot_y + obj_h

                cv2.rectangle(frame, (ox1, oy1), (ox2, oy2), (ob, ob-10, ob-20), -1)
                cv2.rectangle(frame, (ox1, oy1), (ox2, oy2), (ob-40, ob-50, ob-60), 2)
                for ly in range(oy1+8, oy2-4, 10):
                    cv2.line(frame, (ox1+4, ly), (ox2-4, ly), (ob-20, ob-30, ob-40), 1)

        # 프레임 라벨: 화면에 LOADED 후크가 하나라도 있으면 LOADED
        frame_label = 'LOADED' if frame_has_loaded else 'EMPTY'

        # 분진
        n_dust = random.randint(0, 60)
        if n_dust > 0:
            dx = np.random.randint(0, w, n_dust)
            dy = np.random.randint(0, h, n_dust)
            for ddx, ddy in zip(dx, dy):
                cv2.circle(frame, (int(ddx), int(ddy)), 1, (200,200,200), -1)

        label_color = (100, 200, 100) if frame_label == 'LOADED' else (150, 150, 150)
        cv2.putText(frame, f'Frame {i:03d}  {frame_label}',
            (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.5, label_color, 1)

        frames.append(frame)
        labels.append(frame_label)

    return frames, labels


# ── 프레임 로드 ──
if CAM1_MODE == 'folder':
    from pathlib import Path as _Path
    _paths = sorted([str(p) for p in _Path(CAM1_FOLDER_PATH).iterdir()
                     if p.suffix.lower() in ['.jpg','.png','.jpeg']])
    CAM1_FRAMES = [cv2.resize(cv2.imread(p), (CAM1_FRAME_W, CAM1_FRAME_H))
                   for p in _paths[:CAM1_MAX_FRAMES]]
    CAM1_LABELS = ['UNKNOWN'] * len(CAM1_FRAMES)
    print(f'폴더 로드: {len(CAM1_FRAMES)}장')

elif CAM1_MODE == 'video':
    _cap = cv2.VideoCapture(CAM1_VIDEO_PATH)
    CAM1_FRAMES, CAM1_LABELS = [], []
    while len(CAM1_FRAMES) < CAM1_MAX_FRAMES:
        ret, f = _cap.read()
        if not ret: break
        CAM1_FRAMES.append(cv2.resize(f, (CAM1_FRAME_W, CAM1_FRAME_H)))
        CAM1_LABELS.append('UNKNOWN')
    _cap.release()
    print(f'영상 로드: {len(CAM1_FRAMES)}장')

else:  # builtin
    CAM1_FRAMES, CAM1_LABELS = make_cam1_builtin_frames(
        n=CAM1_MAX_FRAMES, w=CAM1_FRAME_W, h=CAM1_FRAME_H,
        loaded_ratio=CAM1_LOADED_RATIO)
    # 저장
    for i, (f, lbl) in enumerate(zip(CAM1_FRAMES, CAM1_LABELS)):
        cv2.imwrite(f'{CAM1_SAVE_DIR}/cam1_frame_{i:03d}.jpg', f)
    print(f'내장 이미지 생성 완료: {len(CAM1_FRAMES)}장 -> {CAM1_SAVE_DIR}')

# 통계
loaded_n = CAM1_LABELS.count('LOADED')
empty_n  = CAM1_LABELS.count('EMPTY')
print(f'LOADED: {loaded_n}장 ({loaded_n/len(CAM1_LABELS)*100:.0f}%)')
print(f'EMPTY : {empty_n}장  ({empty_n/len(CAM1_LABELS)*100:.0f}%)')
print(f'크기  : {CAM1_FRAMES[0].shape}')


In [ ]:
# 섹션 C1-1 — 로드된 프레임 확인
# LOADED/EMPTY 샘플 각 5장씩 표시해 이미지 품질을 확인합니다.

import math as _math

n = len(CAM1_FRAMES)

# LOADED/EMPTY 인덱스 분리
idx_loaded = [i for i,l in enumerate(CAM1_LABELS) if l == 'LOADED']
idx_empty  = [i for i,l in enumerate(CAM1_LABELS) if l == 'EMPTY']

# 각 5장 균등 샘플
n_show  = 5
s_load  = [idx_loaded[i * len(idx_loaded) // n_show]
           for i in range(min(n_show, len(idx_loaded)))]
s_empty = [idx_empty[i * len(idx_empty) // n_show]
           for i in range(min(n_show, len(idx_empty)))]

fig, axes = plt.subplots(2, max(len(s_load), len(s_empty)),
                          figsize=(max(len(s_load),len(s_empty))*3, 6))

for col, idx in enumerate(s_load):
    ax = axes[0][col]
    ax.imshow(cv2.cvtColor(CAM1_FRAMES[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'LOADED\nFrame {idx}', fontsize=9, color='#185FA5')
    ax.axis('off')

for col, idx in enumerate(s_empty):
    ax = axes[1][col]
    ax.imshow(cv2.cvtColor(CAM1_FRAMES[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'EMPTY\nFrame {idx}', fontsize=9, color='#9c9a92')
    ax.axis('off')

plt.suptitle(
    f'[Section C1-1] Camera 1 Frame Check  |  '
    f'Total={n}  LOADED={len(idx_loaded)}  EMPTY={len(idx_empty)}',
    fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('확인:  LOADED — 후크 아래 도장 객체가 보이는가?')
print('       EMPTY  — 아무것도 매달려 있지 않은가?')


---
# 섹션 C1-2. 카메라 1 YOLO 검출

## 클래스 정의

| ID | 클래스 | 설명 |
|----|--------|------|
| 0  | `workpiece` | 후크에 매달린 도장 대상 객체 |

## 모드 선택

| CAM1_YOLO_MODE | 상황 |
|----------------|------|
| `'pretrained'` | 구조 확인용 (workpiece 미지원) |
| `'custom'` | 섹션 C1-2D 완료 후 best.pt 경로 입력 |
| `'manual_roi'` | YOLO 없이 고정 좌표 지정 |

> 파인튜닝이 필요하면 섹션 C1-2B부터 먼저 진행하세요.


In [ ]:
# 섹션 C1-2 — 카메라 1 YOLO 검출 설정

# ── 여기만 수정하세요 ──────────────────────────
CAM1_YOLO_MODE    = 'pretrained'  # 초기값 유지 (파인튜닝 후 'custom'으로 변경)
CAM1_YOLO_WEIGHTS = 'yolov8s.pt'  # custom: 'runs/cam1_finetune/coating_line_cam1/weights/best.pt'
CAM1_YOLO_CONF    = 0.40          # 검출 신뢰도 임계값
                                   # 높을수록 오탐↓ 미검출↑
CAM1_YOLO_CLASSES = None           # custom 모드: [0] (workpiece만)
CAM1_MANUAL_ROI   = (150, 200, 490, 420)  # manual_roi: (x1,y1,x2,y2)

# 카메라 1 클래스 맵 (카메라 2의 CLASS_MAP과 별도)
CAM1_CLASS_MAP = {'workpiece': 0}
# ───────────────────────────────────────────────

from ultralytics import YOLO as _YOLO_C1

cam1_yolo_model = None

if CAM1_YOLO_MODE in ('pretrained', 'custom'):
    _weights = 'yolov8s.pt' if CAM1_YOLO_MODE == 'pretrained' else CAM1_YOLO_WEIGHTS
    cam1_yolo_model = _YOLO_C1(_weights)
    print(f'카메라 1 YOLO 로드: {_weights}')
    if CAM1_YOLO_MODE == 'pretrained':
        print('  COCO 사전학습 — workpiece 클래스 없음 (구조 확인용)')
else:
    print(f'카메라 1 수동 ROI: {CAM1_MANUAL_ROI}')


def cam1_detect_objects(frame):
    if CAM1_YOLO_MODE == 'manual_roi':
        return [{'bbox': CAM1_MANUAL_ROI, 'conf': 1.0, 'cls': 0}]
    results = cam1_yolo_model(
        frame, conf=CAM1_YOLO_CONF,
        classes=CAM1_YOLO_CLASSES, verbose=False)
    return [
        {'bbox': tuple(int(v) for v in box.xyxy[0].tolist()),
         'conf': float(box.conf), 'cls': int(box.cls)}
        for box in results[0].boxes
    ]


def cam1_judge(frame) -> str:
    # 도장 객체(workpiece) 검출 여부로 LOADED/EMPTY 판정
    dets = cam1_detect_objects(frame)
    return 'LOADED' if len(dets) > 0 else 'EMPTY'


# 검출 테스트
_test_dets = cam1_detect_objects(CAM1_FRAMES[0])
print(f'검출 테스트 (Frame 0): {len(_test_dets)}개')


---
# 섹션 C1-2B. 카메라 1 — Grounding DINO 자동 라벨링

## 역할

카메라 2의 섹션 2-B와 동일한 방식으로 카메라 1 이미지를 라벨링합니다.
프롬프트를 도장 객체에 맞게 변경하여 bbox를 자동 생성합니다.

## 클래스 정의

| ID | 클래스 | 설명 |
|----|--------|------|
| 0  | `workpiece` | 후크에 매달린 도장 대상 |

> LOADED 프레임만 라벨링합니다. EMPTY 프레임은 라벨 파일을 빈 파일로 저장합니다.


In [ ]:
# 섹션 C1-2B — 카메라 1 Grounding DINO 설정

# ── 여기만 수정하세요 ──────────────────────────
CAM1_GDINO_PROMPT         = 'rectangular object . hanging object . workpiece'
CAM1_GDINO_BOX_THRESHOLD  = 0.30
CAM1_GDINO_TEXT_THRESHOLD = 0.15
CAM1_MAX_BOX_RATIO        = 0.50  # 이미지 대비 bbox 최대 크기 비율
CAM1_USE_NMS              = False
CAM1_NMS_IOU_THR          = 0.5
# ───────────────────────────────────────────────

# 카메라 1 이미지 경로 목록 구성
# LOADED 프레임만 라벨링 대상 (EMPTY는 빈 라벨)
cam1_image_paths = []
for i, (frame, label) in enumerate(zip(CAM1_FRAMES, CAM1_LABELS)):
    dst = f'dataset_cam1/images/train/cam1_frame_{i:03d}.jpg'
    if not os.path.exists(dst):
        cv2.imwrite(dst, frame)
    cam1_image_paths.append(dst)

print(f'카메라 1 이미지 경로 준비: {len(cam1_image_paths)}장')
print(f'  LOADED: {CAM1_LABELS.count("LOADED")}장 (라벨링 대상)')
print(f'  EMPTY : {CAM1_LABELS.count("EMPTY")}장 (빈 라벨 저장)')

# EMPTY 프레임은 미리 빈 라벨 파일 생성
for i, label in enumerate(CAM1_LABELS):
    if label == 'EMPTY':
        lp = f'dataset_cam1/labels/train/cam1_frame_{i:03d}.txt'
        open(lp, 'w').close()  # 빈 파일

print('EMPTY 프레임 빈 라벨 파일 생성 완료')


In [ ]:
# 섹션 C1-2B — Grounding DINO 추론 + 라벨 저장
# 카메라 2의 섹션 2-B와 동일한 구조입니다.
# LOADED 프레임에만 GDINO를 실행하고, 도장 객체 bbox를 라벨로 저장합니다.
#
# ※ gdino_processor / gdino_model_hf 가 없으면 자동으로 로드합니다.
#   카메라 2 섹션 2-B를 먼저 실행했다면 이미 로드되어 있으므로 건너뜁니다.

import numpy as np
from PIL import Image as _PIL_c1
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch as _torch_c1

# ── Grounding DINO 자동 로드 ──
# 카메라 2 섹션 2-B에서 이미 로드했으면 재사용합니다.
# 카메라 1 파이프라인만 단독으로 실행할 때도 자동으로 로드됩니다.
if 'gdino_processor' not in dir() or 'gdino_model_hf' not in dir():
    print('Grounding DINO 로드 중... (첫 실행 시 약 1GB 다운로드)')
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    _gdino_repo = 'IDEA-Research/grounding-dino-base'
    device_gdino = 'cuda' if _torch_c1.cuda.is_available() else 'cpu'
    gdino_processor = AutoProcessor.from_pretrained(_gdino_repo)
    gdino_model_hf  = AutoModelForZeroShotObjectDetection.from_pretrained(
        _gdino_repo).to(device_gdino)
    print(f'Grounding DINO 로드 완료  디바이스: {device_gdino}')
else:
    print('Grounding DINO 이미 로드됨 — 재사용합니다.')

device_gdino = 'cuda' if _torch_c1.cuda.is_available() else 'cpu'

def cam1_nms_boxes(boxes, scores, iou_threshold=0.5):
    if len(boxes) == 0:
        return []
    boxes_arr  = np.array(boxes,  dtype=np.float32)
    scores_arr = np.array(scores, dtype=np.float32)
    order = scores_arr.argsort()[::-1]
    keep  = []
    while len(order) > 0:
        i = order[0]; keep.append(i)
        if len(order) == 1: break
        rest = order[1:]
        xx1 = np.maximum(boxes_arr[i,0], boxes_arr[rest,0])
        yy1 = np.maximum(boxes_arr[i,1], boxes_arr[rest,1])
        xx2 = np.minimum(boxes_arr[i,2], boxes_arr[rest,2])
        yy2 = np.minimum(boxes_arr[i,3], boxes_arr[rest,3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        area_i    = (boxes_arr[i,2]-boxes_arr[i,0])*(boxes_arr[i,3]-boxes_arr[i,1])
        area_rest = ((boxes_arr[rest,2]-boxes_arr[rest,0])*
                     (boxes_arr[rest,3]-boxes_arr[rest,1]))
        iou = inter / (area_i + area_rest - inter + 1e-6)
        order = rest[iou < iou_threshold]
    return keep


def run_cam1_gdino(img_path: str, frame_label: str):
    # EMPTY 프레임은 추론 없이 빈 라벨 반환
    if frame_label == 'EMPTY':
        pil_img = _PIL_c1.open(img_path).convert('RGB')
        return [], pil_img

    pil_img = _PIL_c1.open(img_path).convert('RGB')
    W, H    = pil_img.size
    text    = CAM1_GDINO_PROMPT.replace('.', '. ').strip() + '.'

    inputs  = gdino_processor(images=pil_img, text=text,
                               return_tensors='pt').to(device_gdino)
    with _torch_c1.no_grad():
        outputs = gdino_model_hf(**inputs)

    res = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold      = CAM1_GDINO_BOX_THRESHOLD,
        text_threshold = CAM1_GDINO_TEXT_THRESHOLD,
        target_sizes   = [(H, W)]
    )[0]

    raw_boxes, raw_scores = [], []
    for score, label, box in zip(res['scores'], res['text_labels'], res['boxes']):
        raw_boxes.append(box.tolist())
        raw_scores.append(float(score))

    if CAM1_USE_NMS and raw_boxes:
        kept = cam1_nms_boxes(raw_boxes, raw_scores, CAM1_NMS_IOU_THR)
    else:
        kept = list(range(len(raw_boxes)))

    final = []
    for i in kept:
        x1, y1, x2, y2 = raw_boxes[i]
        bw_r = (x2-x1)/W; bh_r = (y2-y1)/H
        if bw_r > CAM1_MAX_BOX_RATIO or bh_r > CAM1_MAX_BOX_RATIO:
            continue
        cx = ((x1+x2)/2)/W; cy = ((y1+y2)/2)/H
        final.append((0, cx, cy, bw_r, bh_r, raw_scores[i], raw_boxes[i]))

    return final, pil_img


# ── 전체 이미지 추론 + 라벨 저장 ──
os.makedirs('dataset_cam1/labels/train', exist_ok=True)

total_boxes = 0
loaded_with_box = 0
loaded_no_box   = 0

n_imgs = len(cam1_image_paths)
n_cols = min(3, n_imgs)
n_rows = (n_imgs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
axes = np.array(axes).flatten()

for img_idx, (img_path, frame_label) in enumerate(
        zip(cam1_image_paths, CAM1_LABELS)):

    results, pil_img = run_cam1_gdino(img_path, frame_label)

    # 라벨 저장
    lp = f'dataset_cam1/labels/train/{Path(img_path).stem}.txt'
    lines = [f'{r[0]} {r[1]:.6f} {r[2]:.6f} {r[3]:.6f} {r[4]:.6f}'
             for r in results]
    open(lp, 'w').write('\n'.join(lines))

    if frame_label == 'LOADED':
        if len(results) > 0:
            loaded_with_box += 1
        else:
            loaded_no_box += 1
    total_boxes += len(results)

    ax = axes[img_idx]
    ax.imshow(pil_img)
    for r in results:
        x1,y1,x2,y2 = r[6]
        ax.add_patch(patches.Rectangle(
            (x1,y1), x2-x1, y2-y1, lw=2, edgecolor='#EF9F27', facecolor='none'))
        ax.text(x1, max(y1-5,10), f'workpiece {r[5]:.2f}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#EF9F27', alpha=0.85))
    status_str = f'{frame_label} | {len(results)} box'
    ax.set_title(f'{Path(img_path).name}\n{status_str}', fontsize=8)
    ax.axis('off')

for ax in axes[n_imgs:]: ax.axis('off')
plt.suptitle(
    f'[Section C1-2B] Camera 1 GDINO Result\n'
    f"prompt='{CAM1_GDINO_PROMPT}'  box={CAM1_GDINO_BOX_THRESHOLD}",
    fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'라벨링 완료: 총 {total_boxes}개 박스')
print(f'  LOADED + 박스 있음: {loaded_with_box}장')
print(f'  LOADED + 박스 없음: {loaded_no_box}장 (임계값 낮추거나 프롬프트 수정 필요)')
print(f'  EMPTY (빈 라벨):    {CAM1_LABELS.count("EMPTY")}장')


---
# 섹션 C1-2C. 카메라 1 — 라벨 검수

자동 생성된 라벨이 맞는지 시각적으로 확인합니다.
LOADED 프레임에서 도장 객체 bbox가 올바르게 감싸고 있는지 확인하세요.


In [ ]:
# 섹션 C1-2C — 카메라 1 라벨 검수 시각화

def get_cam1_label_path(img_path):
    return f'dataset_cam1/labels/train/{Path(img_path).stem}.txt'

check_paths = cam1_image_paths[::max(1, len(cam1_image_paths)//12)][:12]
n_cols = 3
n_rows = (len(check_paths) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
axes = np.array(axes).flatten()

for ax, img_path in zip(axes, check_paths):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h_px, w_px = img.shape[:2]
    ax.imshow(img)
    lp = get_cam1_label_path(img_path)
    n_boxes = 0
    if os.path.exists(lp):
        for line in open(lp).read().strip().splitlines():
            if not line.strip(): continue
            cls_id = int(line.split()[0])
            cx,cy,bw,bh = [float(x) for x in line.split()[1:5]]
            x1=(cx-bw/2)*w_px; y1=(cy-bh/2)*h_px
            ax.add_patch(patches.Rectangle(
                (x1,y1), bw*w_px, bh*h_px, lw=2, ec='#EF9F27', fc='none'))
            ax.text(x1, y1-4, 'workpiece',
                    color='#EF9F27', fontsize=9, fontweight='bold')
            n_boxes += 1
    # 라벨에 해당하는 원본 라벨 찾기
    fi = cam1_image_paths.index(img_path)
    orig_label = CAM1_LABELS[fi] if fi < len(CAM1_LABELS) else '?'
    ax.set_title(
        f'{Path(img_path).name}\n{orig_label} | {n_boxes} box(es)',
        fontsize=9,
        color='#185FA5' if orig_label == 'LOADED' else '#9c9a92')
    ax.axis('off')

for ax in axes[len(check_paths):]: ax.axis('off')
plt.suptitle('[Section C1-2C] Camera 1 Label Review  |  Orange=workpiece',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

# 통계
total_boxes = 0
for img_path in cam1_image_paths:
    lp = get_cam1_label_path(img_path)
    if not os.path.exists(lp): continue
    for line in open(lp).read().strip().splitlines():
        if line.strip(): total_boxes += 1

print(f'이미지: {len(cam1_image_paths)}장  총 박스: {total_boxes}개')
print(f'  LOADED: {CAM1_LABELS.count("LOADED")}장  EMPTY: {CAM1_LABELS.count("EMPTY")}장')
print()
print('확인: bbox가 도장 객체를 정확히 감쌈 / EMPTY 프레임에 박스 없음')


---
# 섹션 C1-2D. 카메라 1 — YOLOv8 파인튜닝

카메라 2(섹션 2-D)와 동일한 방식으로 파인튜닝합니다.
클래스가 `workpiece` 1개이며 저장 경로가 `dataset_cam1/`로 분리됩니다.

| 파라미터 | 기본값 | 조정 기준 |
|----------|--------|----------|
| `CAM1_EPOCHS` | 80 | 이미지 수에 따라 조정 |
| `CAM1_BATCH` | 8 | GPU 메모리에 따라 조정 |


In [ ]:
# 섹션 C1-2D — 카메라 1 YOLOv8 파인튜닝

# ── 여기만 수정하세요 ──────────────────────────
CAM1_EPOCHS      = 80
CAM1_BATCH       = 8
CAM1_IMGSZ       = 640
CAM1_LR          = 0.001
CAM1_FREEZE      = 10
CAM1_BASE_MODEL  = 'yolov8s.pt'
CAM1_RANDOM_SEED = 42
# ───────────────────────────────────────────────

from ultralytics import YOLO as _YOLO_C1_ft
import torch as _torch_c1_ft

# ── 데이터 존재 확인 ──
_all_c1 = (glob.glob('dataset_cam1/images/train/*.jpg') +
           glob.glob('dataset_cam1/images/train/*.png'))

if len(_all_c1) == 0:
    print('dataset_cam1/images/train 에 이미지가 없습니다.')
    print('섹션 C1-2B를 먼저 실행하세요.')
    raise RuntimeError('이미지 없음')

# ── train/val 분할 ──
random.seed(CAM1_RANDOM_SEED)
random.shuffle(_all_c1)
_val_n   = max(1, int(len(_all_c1) * 0.2))
_val_imgs = _all_c1[:_val_n]

os.makedirs('dataset_cam1/images/val', exist_ok=True)
os.makedirs('dataset_cam1/labels/val', exist_ok=True)
for _ip in _val_imgs:
    _lp = f'dataset_cam1/labels/train/{Path(_ip).stem}.txt'
    shutil.move(_ip, f'dataset_cam1/images/val/{Path(_ip).name}')
    if os.path.exists(_lp):
        shutil.move(_lp, f'dataset_cam1/labels/val/{Path(_ip).stem}.txt')

print(f'데이터 분할  train:{len(_all_c1)-_val_n}장  val:{_val_n}장')

# ── data.yaml 생성 ──
_cam1_yaml = {
    'path':  '/content/dataset_cam1',
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': {0: 'workpiece'},
}
with open('cam1_data.yaml', 'w') as yf:
    yaml.dump(_cam1_yaml, yf, allow_unicode=True)
print(f'cam1_data.yaml 생성: {_cam1_yaml["names"]}')

# ── 파인튜닝 실행 ──
print(f'\n카메라 1 파인튜닝 시작')
print(f'  베이스: {CAM1_BASE_MODEL}  에포크: {CAM1_EPOCHS}')
print(f'  GPU: {"사용" if _torch_c1_ft.cuda.is_available() else "없음 (CPU)"}')

_c1_ft = _YOLO_C1_ft(CAM1_BASE_MODEL)
_c1_ft.train(
    data='cam1_data.yaml', epochs=CAM1_EPOCHS, batch=CAM1_BATCH,
    imgsz=CAM1_IMGSZ, lr0=CAM1_LR, freeze=CAM1_FREEZE, patience=15,
    augment=True, degrees=10, fliplr=0.5, mosaic=0.8,
    project='runs/cam1_finetune', name='coating_line_cam1',
    exist_ok=True, verbose=False,
)

CAM1_FINETUNED_PT = 'runs/cam1_finetune/coating_line_cam1/weights/best.pt'
print(f'\n카메라 1 파인튜닝 완료 -> {CAM1_FINETUNED_PT}')
print('섹션 C1-2E에서 성능을 검증하세요.')


---
# 섹션 C1-2E. 카메라 1 — 성능 검증

| mAP@50 | 판정 | 조치 |
|--------|------|------|
| 0.70↑ | GOOD | 섹션 C1-3으로 진행 |
| 0.50~0.70 | FAIR | 사용 가능, 이미지 추가 권장 |
| 0.50↓ | LOW | 이미지 추가 후 재실행 |


In [ ]:
# 섹션 C1-2E — 카메라 1 성능 검증

import glob as _gl_c1e
from ultralytics import YOLO as _YOLO_C1_val
import pandas as pd

_c1_candidates = _gl_c1e.glob('runs/cam1_finetune/**/best.pt', recursive=True)
if not _c1_candidates:
    raise FileNotFoundError('best.pt 없음 — 섹션 C1-2D를 먼저 실행하세요.')

CAM1_FINETUNED_PT = max(_c1_candidates, key=os.path.getmtime)
print(f'검증 모델: {CAM1_FINETUNED_PT}')

_c1_val = _YOLO_C1_val(CAM1_FINETUNED_PT)
_c1_met = _c1_val.val(data='cam1_data.yaml', verbose=False)
_c1_map50 = _c1_met.box.map50

print('=' * 50)
print(f'  mAP@50    : {_c1_map50:.3f}  (0.70 이상 권장)')
print(f'  mAP@50-95 : {_c1_met.box.map:.3f}')
print(f'  Precision : {_c1_met.box.mp:.3f}')
print(f'  Recall    : {_c1_met.box.mr:.3f}')
print('=' * 50)

if   _c1_map50 >= 0.70: print('판정: GOOD — 섹션 C1-3으로 진행하세요.')
elif _c1_map50 >= 0.50: print('판정: FAIR — 사용 가능, 이미지 추가 권장')
else:                    print('판정: LOW  — 이미지 추가 후 재실행')

# 학습 곡선 시각화
_c1_csv = _gl_c1e.glob('runs/cam1_finetune/**/results.csv', recursive=True)
if _c1_csv:
    df = pd.read_csv(max(_c1_csv, key=os.path.getmtime))
    df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    def _fc(df, kw):
        for c in df.columns:
            if all(k.lower() in c.lower() for k in kw): return c
        return None

    ep = df['epoch'] if 'epoch' in df.columns else range(len(df))
    c_map50 = _fc(df, ['map50'])

    _tb = _fc(df, ['train','box']); _tc = _fc(df, ['train','cls'])
    if _tb and _tc:
        axes[0].plot(ep, df[_tb], label='box loss', color='#EF9F27')
        axes[0].plot(ep, df[_tc], label='cls loss', color='#185FA5')
        axes[0].set_title('Camera 1 Train Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    if c_map50:
        axes[1].plot(ep, df[c_map50], color='#0F6E56', lw=2, label='mAP@50')
        axes[1].axhline(0.70, color='gray', ls='--', label='0.70 target')
        axes[1].set_title('Camera 1 mAP@50'); axes[1].legend(); axes[1].grid(alpha=0.3)
        axes[1].set_ylim(0, 1.05)

    plt.suptitle('[Section C1-2E] Camera 1 Training Curves', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()


---
# 섹션 C1-3. 카메라 1 — LOADED/EMPTY 판정 및 후크 큐 연동

파인튜닝 완료 후 CAM1_FINETUNED_PT 모델로 전체 프레임을 판정하고
결과를 섹션 7~8의 후크 상태 큐에 연동합니다.


In [ ]:
# 섹션 C1-3 — 카메라 1 파인튜닝 모델 적용 + 전체 프레임 LOADED/EMPTY 판정
#
# 1) 파인튜닝 모델(CAM1_FINETUNED_PT)을 자동으로 로드합니다.
# 2) 전체 CAM1_FRAMES에 대해 LOADED/EMPTY를 판정합니다.
# 3) 결과를 CAM1_RESULTS 딕셔너리에 저장합니다.
# 4) 섹션 8에서 이 딕셔너리를 후크 큐에 push합니다.

import glob as _gl_c13
from ultralytics import YOLO as _YOLO_C1_run

# ── 여기만 수정하세요 ──────────────────────────
CAM1_CONF_FINAL = 0.40  # 최종 판정 신뢰도 임계값
# ───────────────────────────────────────────────

# ── 파인튜닝 모델 자동 탐지 ──
if 'CAM1_FINETUNED_PT' not in dir() or not os.path.exists(CAM1_FINETUNED_PT):
    _c1_cands = _gl_c13.glob('runs/cam1_finetune/**/best.pt', recursive=True)
    if not _c1_cands:
        print('best.pt 없음 — 섹션 C1-2D를 먼저 실행하세요.')
        raise RuntimeError('카메라 1 모델 없음')
    CAM1_FINETUNED_PT = max(_c1_cands, key=os.path.getmtime)

_c1_model_run = _YOLO_C1_run(CAM1_FINETUNED_PT)
print(f'카메라 1 모델 로드: {CAM1_FINETUNED_PT}')
print(f'클래스: {_c1_model_run.names}')
print()

# ── 전체 프레임 판정 ──
CAM1_RESULTS = {}
loaded_n = empty_n = 0

for fi, frame in enumerate(CAM1_FRAMES):
    results = _c1_model_run(frame, conf=CAM1_CONF_FINAL, verbose=False)
    status  = 'LOADED' if len(results[0].boxes) > 0 else 'EMPTY'
    CAM1_RESULTS[fi] = status
    if status == 'LOADED': loaded_n += 1
    else:                  empty_n  += 1

print(f'카메라 1 판정 완료: {len(CAM1_RESULTS)}프레임')
print(f'  LOADED: {loaded_n}프레임 ({loaded_n/len(CAM1_FRAMES)*100:.1f}%)')
print(f'  EMPTY : {empty_n}프레임  ({empty_n/len(CAM1_FRAMES)*100:.1f}%)')
print()

# 판정 결과 시각화
fig, ax = plt.subplots(figsize=(14, 2.5))
fis_l = [fi for fi,st in CAM1_RESULTS.items() if st=='LOADED']
fis_e = [fi for fi,st in CAM1_RESULTS.items() if st=='EMPTY']
ax.bar(fis_l, [1]*len(fis_l), color='#185FA5', alpha=0.8, label=f'LOADED ({loaded_n})')
ax.bar(fis_e, [1]*len(fis_e), color='#9c9a92', alpha=0.5, label=f'EMPTY  ({empty_n})')
ax.set_xlabel('Frame'); ax.set_yticks([])
ax.set_title('[Section C1-3] Camera 1 Final Judgement  |  Blue=LOADED  Gray=EMPTY')
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

# 내장 라벨과 비교 (builtin 모드일 때)
if CAM1_MODE == 'builtin':
    correct = sum(1 for fi in range(len(CAM1_FRAMES))
                  if CAM1_RESULTS.get(fi) == CAM1_LABELS[fi])
    acc = correct / len(CAM1_FRAMES) * 100
    print(f'내장 라벨 대비 정확도: {correct}/{len(CAM1_FRAMES)} ({acc:.1f}%)')
    if acc >= 80:
        print('결과: 양호 — 섹션 8 통합 실행으로 진행하세요.')
    else:
        print('결과: 미흡 — CAM1_CONF_FINAL 조정 또는 재파인튜닝을 권장합니다.')

print()
print('CAM1_RESULTS 준비 완료.')
print('섹션 8 통합 실행 셀에서 이 결과가 자동으로 후크 큐에 연동됩니다.')
